## Data paths

Private Colab and Google Drive paths were replaced with repository-relative `data/` paths. Place permitted local inputs under `data/` or adapt the path variables for your environment. Original private research data are not included.

# Converting PDF files into TXT files

## Notebook overview

This notebook documents the **Arabic PDF-to-text preparation pipeline** used in the project. The workflow begins with document-format conversion, then applies several rounds of Arabic text repair and quality control, and finally prepares the resulting files for OCR-based trimming and AI-detection workflows.

**Main workflow**
1. Convert the source material into editable text-friendly formats.
2. Repair known Arabic character and paragraph-structure issues.
3. Detect heavily corrupted files and repair minor corruption automatically.
4. Convert difficult PDFs into scanned PDFs and OCR text when needed.
5. Route the cleaned files into later trimming and AI-detection steps.

Before we begin, we removed the first page of each file if it contained less than 50 words, since those are cover pages that are not a part of the assignment text we mean to detect.

We converted the PDF files to DOCX format using the online tool ILovePDF.

## Fixing an Arabic Character Issue

### Code block 1 — Repair a recurring Arabic character issue in DOCX files

This block recursively edits the converted DOCX files to replace the character **ھ** with **ه**. It standardizes one recurring OCR/conversion issue before the DOCX files are exported to plain text.

In [ ]:
import os
import shutil
import tempfile
import zipfile

# Configure these environment variables before running:
# DOCX_ROOT_FOLDER: directory containing DOCX files to update
# DOCX_BACKUP_FOLDER: separate directory for an optional full backup
ROOT_FOLDER = os.environ["DOCX_ROOT_FOLDER"]
MAKE_BACKUP = True
BACKUP_FOLDER = os.environ["DOCX_BACKUP_FOLDER"]
OLD_CHAR = "ھ"
NEW_CHAR = "ه"

if not os.path.isdir(ROOT_FOLDER):
    raise NotADirectoryError(f"DOCX root folder does not exist: {ROOT_FOLDER}")

if MAKE_BACKUP:
    root_path = os.path.abspath(ROOT_FOLDER)
    backup_path = os.path.abspath(BACKUP_FOLDER)

    if backup_path == root_path:
        raise ValueError("BACKUP_FOLDER must differ from ROOT_FOLDER.")
    if os.path.commonpath([root_path, backup_path]) == root_path:
        raise ValueError("BACKUP_FOLDER must not be inside ROOT_FOLDER.")

    if not os.path.exists(backup_path):
        print(f"Creating backup:\n  {backup_path}")
        shutil.copytree(root_path, backup_path)
    else:
        print(f"Backup folder already exists, skipping backup:\n  {backup_path}")


def patch_docx_in_place(docx_path, old_char=OLD_CHAR, new_char=NEW_CHAR):
    """Replace matching text in UTF-8 XML parts without rebuilding the DOCX."""
    tmp_fd, tmp_zip_path = tempfile.mkstemp(suffix=".docx")
    os.close(tmp_fd)

    replaced_count = 0
    changed = False

    try:
        with (
            zipfile.ZipFile(docx_path, "r") as source,
            zipfile.ZipFile(
                tmp_zip_path, "w", compression=zipfile.ZIP_DEFLATED
            ) as destination,
        ):
            for item in source.infolist():
                data = source.read(item.filename)

                # DOCX text is stored in XML; retain non-XML package parts unchanged.
                if item.filename.endswith(".xml"):
                    try:
                        text = data.decode("utf-8")
                    except UnicodeDecodeError:
                        pass
                    else:
                        count = text.count(old_char)
                        if count:
                            data = text.replace(old_char, new_char).encode("utf-8")
                            replaced_count += count
                            changed = True

                destination.writestr(item, data)

        if changed:
            shutil.move(tmp_zip_path, docx_path)
        else:
            os.remove(tmp_zip_path)

        return changed, replaced_count, None

    except Exception as error:
        if os.path.exists(tmp_zip_path):
            os.remove(tmp_zip_path)
        return False, 0, str(error)


total_files = 0
changed_files = 0
total_replacements = 0
failed_files = []

for root, _, files in os.walk(ROOT_FOLDER):
    for filename in files:
        if filename.lower().endswith(".docx"):
            total_files += 1
            docx_path = os.path.join(root, filename)
            changed, count, error = patch_docx_in_place(docx_path)

            if error is not None:
                failed_files.append((docx_path, error))
                print(f"FAILED: {docx_path}\n  Error: {error}")
            elif changed:
                changed_files += 1
                total_replacements += count
                print(f"UPDATED: {docx_path} | replacements={count}")
            else:
                print(f"NO CHANGE: {docx_path}")

print("\n" + "=" * 60)
print("DONE")
print(f"Total DOCX files scanned: {total_files}")
print(f"Files changed: {changed_files}")
print(f"Total replacements made: {total_replacements}")
print(f"Files failed: {len(failed_files)}")

if failed_files:
    print("\nFAILED FILES:")
    for docx_path, error in failed_files:
        print(f"- {docx_path}\n  {error}")

## Convert DOCX into TXT

### Code block 2 — Export DOCX files to TXT while preserving folder structure

This block uploads each DOCX file as a Google Doc and exports it as TXT. The goal is to mimic a reliable manual export workflow while preserving the original subfolder structure in Drive.

In [ ]:
# Convert DOCX files through Google Docs and export them as plain text.
# Relative paths under SOURCE_ROOT are preserved in OUTPUT_ROOT.

!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib

import os
from pathlib import Path

from google.colab import auth, drive
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

# Set these environment variables to Drive-mounted or other accessible directories.
SOURCE_ROOT = Path(os.environ["DOCX_SOURCE_ROOT"]).expanduser()
OUTPUT_ROOT = Path(os.environ["TXT_OUTPUT_ROOT"]).expanduser()

SKIP_EXISTING_TXT = True
DELETE_TEMP_GOOGLE_DOCS = True

drive.mount("/content/drive")
auth.authenticate_user()

creds, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=creds)


def upload_docx_as_google_doc(docx_path):
    """Upload a DOCX as a temporary Google Doc and return its Drive ID."""
    metadata = {
        "name": docx_path.name,
        "mimeType": "application/vnd.google-apps.document",
    }
    media = MediaFileUpload(
        str(docx_path),
        mimetype="application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        resumable=True,
    )
    created = drive_service.files().create(
        body=metadata,
        media_body=media,
        fields="id",
    ).execute()
    return created["id"]


def export_google_doc_as_txt(file_id, output_txt_path):
    """Export a Google Doc as plain text."""
    request = drive_service.files().export_media(
        fileId=file_id,
        mimeType="text/plain",
    )
    output_txt_path.parent.mkdir(parents=True, exist_ok=True)

    with output_txt_path.open("wb") as output_file:
        downloader = MediaIoBaseDownload(output_file, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def delete_drive_file(file_id):
    drive_service.files().delete(fileId=file_id).execute()


def convert_docx_to_txt(docx_path, txt_path):
    temp_gdoc_id = None
    try:
        temp_gdoc_id = upload_docx_as_google_doc(docx_path)
        export_google_doc_as_txt(temp_gdoc_id, txt_path)
        if DELETE_TEMP_GOOGLE_DOCS:
            delete_drive_file(temp_gdoc_id)
        return True, None
    except Exception as exc:
        if temp_gdoc_id and DELETE_TEMP_GOOGLE_DOCS:
            try:
                delete_drive_file(temp_gdoc_id)
            except Exception:
                pass
        return False, str(exc)


total_docx = 0
converted = 0
skipped = 0
failed = []

for root, _, files in os.walk(SOURCE_ROOT):
    for filename in files:
        if not filename.lower().endswith(".docx"):
            continue

        total_docx += 1
        docx_path = Path(root) / filename
        relative_path = docx_path.relative_to(SOURCE_ROOT)
        txt_path = OUTPUT_ROOT / relative_path.with_suffix(".txt")

        if SKIP_EXISTING_TXT and txt_path.exists():
            skipped += 1
            print(f"SKIPPED (exists): {txt_path}")
            continue

        success, error = convert_docx_to_txt(docx_path, txt_path)
        if success:
            converted += 1
            print(f"CONVERTED: {docx_path} -> {txt_path}")
        else:
            failed.append((docx_path, error))
            print(f"FAILED: {docx_path}\n  Error: {error}")

print("\n" + "=" * 70)
print("DONE")
print(f"Total DOCX found:   {total_docx}")
print(f"Converted:          {converted}")
print(f"Skipped existing:   {skipped}")
print(f"Failed:             {len(failed)}")

if failed:
    print("\nFAILED FILES:")
    for path, error in failed:
        print(f"- {path}\n  {error}")

# Cleaning TXT Files

### Code block 3 — Repair paragraph structure in Arabic TXT files

This block removes false line breaks inside Arabic paragraphs while keeping genuine paragraph breaks, bullets, and headings. It is designed to reduce formatting noise that could hurt downstream AI-detection quality.

In [ ]:
import os
import re

INPUT_ROOT = os.environ["ARABIC_TXT_INPUT_ROOT"]
OUTPUT_ROOT = os.environ["ARABIC_TXT_OUTPUT_ROOT"]


def normalize_newlines(text):
    return text.replace("\r\n", "\n").replace("\r", "\n").replace("\ufeff", "")


def is_blank(line):
    return not line.strip()


def is_numbered_or_list_line(line):
    s = line.strip()
    patterns = [
        r"^\d+\s*-\s*$",
        r"^\d+\s*-\s+.+$",
        r"^\.\d+\s+.+$",
        r"^\d+\.\s+.+$",
        r"^[٠-٩]+\s*-\s*.+$",
        r"^[٠-٩]+\.\s+.+$",
        r"^[أ-ي]\s*[\.\-]\s+.+$",
        r"^[A-Za-z]\s*[\.\-]\s+.+$",
        r"^[\-*\•\∙]\s+.+$",
    ]
    return any(re.match(pattern, s) for pattern in patterns)


def is_short_heading(line):
    s = line.strip()
    if not s or len(s) > 60 or is_numbered_or_list_line(s):
        return False
    return not re.search(r"[.!؟?؛;:]$", s)


def starts_new_structural_unit(line):
    return is_numbered_or_list_line(line) or is_short_heading(line)


def clean_spaces(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"


def split_into_blocks(lines):
    """Split text at blank lines while retaining nonblank line groups."""
    blocks = []
    current = []

    for line in lines:
        if is_blank(line):
            if current:
                blocks.append(current)
                current = []
        else:
            current.append(line.rstrip())

    if current:
        blocks.append(current)

    return blocks


def repair_block(block):
    """Join wrapped paragraph lines while retaining headings and list items."""
    if not block:
        return ""

    paragraphs = []
    current_paragraph = []

    for index, line in enumerate(block):
        line = line.strip()

        if index == 0:
            current_paragraph.append(line)
        elif starts_new_structural_unit(line):
            paragraphs.append(" ".join(current_paragraph).strip())
            current_paragraph = [line]
        else:
            current_paragraph.append(line)

    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph).strip())

    return "\n".join(paragraphs)


def repair_text(text):
    lines = [line.rstrip() for line in normalize_newlines(text).split("\n")]
    blocks = split_into_blocks(lines)
    repaired = "\n\n".join(repair_block(block) for block in blocks if block)
    return clean_spaces(repaired)


total_files = 0
written_files = 0
failed_files = []

for root, _, files in os.walk(INPUT_ROOT):
    for filename in files:
        if not filename.lower().endswith(".txt"):
            continue

        total_files += 1
        input_path = os.path.join(root, filename)
        relative_path = os.path.relpath(input_path, INPUT_ROOT)
        output_path = os.path.join(OUTPUT_ROOT, relative_path)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        try:
            with open(input_path, "r", encoding="utf-8") as file:
                repaired = repair_text(file.read())

            with open(output_path, "w", encoding="utf-8") as file:
                file.write(repaired)

            written_files += 1
            print(f"FIXED: {input_path} -> {output_path}")
        except Exception as error:
            failed_files.append((input_path, str(error)))
            print(f"FAILED: {input_path}\n  Error: {error}")

print("\n" + "=" * 60)
print("DONE")
print(f"Total TXT files found: {total_files}")
print(f"Fixed TXT files written: {written_files}")
print(f"Failed files: {len(failed_files)}")

if failed_files:
    print("\nFAILED FILES:")
    for path, error in failed_files:
        print(f"- {path}\n  {error}")

### Code block 4 — Flag heavily corrupted TXT files and auto-fix minor corruption

This block scans the Arabic TXT corpus, identifies files that appear too damaged for safe automatic use, and applies limited automatic fixes to less severe corruption in the remaining files.

In [ ]:
import os
import re
import csv
import shutil
import unicodedata

# Configure these paths outside the source corpus. Private research data is not included.
INPUT_ROOT = os.environ["ARABIC_TEXT_INPUT_ROOT"]
OUTPUT_ROOT = os.environ["ARABIC_TEXT_OUTPUT_ROOT"]
MAJOR_ROOT = os.environ["ARABIC_TEXT_MAJOR_ROOT"]
REPORT_DIR = os.environ["ARABIC_TEXT_REPORT_DIR"]

OVERWRITE_OUTPUT = True
USE_OPENAI_FOR_SUSPICIOUS_LINES = False
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
OPENAI_MODEL = "gpt-5-mini"
MAX_SUSPICIOUS_LINES_PER_FILE_FOR_OPENAI = 12

MAJOR_MIN_ARABIC_CHAR_RATIO = 0.45
MAJOR_MAX_SUSPICIOUS_TOKEN_RATIO = 0.35
MAJOR_MAX_NON_ARABIC_WORD_RATIO = 0.30


def is_within(path, parent):
    return os.path.commonpath([path, parent]) == parent


def validate_paths():
    input_root = os.path.abspath(INPUT_ROOT)
    destinations = {
        "ARABIC_TEXT_OUTPUT_ROOT": os.path.abspath(OUTPUT_ROOT),
        "ARABIC_TEXT_MAJOR_ROOT": os.path.abspath(MAJOR_ROOT),
        "ARABIC_TEXT_REPORT_DIR": os.path.abspath(REPORT_DIR),
    }

    if not os.path.isdir(input_root):
        raise FileNotFoundError(f"Input directory does not exist: {input_root}")

    if len(set(destinations.values())) != len(destinations):
        raise ValueError("Output, major-copy, and report directories must be distinct.")

    for name, path in destinations.items():
        if is_within(path, input_root):
            raise ValueError(f"{name} must be outside ARABIC_TEXT_INPUT_ROOT.")


validate_paths()
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(MAJOR_ROOT, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

ARABIC_RANGES = (
    "\u0600-\u06ff" "\u0750-\u077f" "\u08a0-\u08ff" "\ufb50-\ufdff" "\ufe70-\ufeff"
)

ARABIC_CHAR_RE = re.compile(f"[{ARABIC_RANGES}]")
EN_CHAR_RE = re.compile(r"[A-Za-z]")
WORD_RE = re.compile(r"\S+")
HARAKAT_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL_RE = re.compile(r"\u0640")
INVISIBLES_RE = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069\uFEFF]")
SPACE_BEFORE_PUNCT_RE = re.compile(r"\s+([،؛:!?？\?\.\,\)\]\}])")
SPACE_AFTER_OPEN_RE = re.compile(r"([\(\[\{])\s+")
MULTISPACE_RE = re.compile(r"[ \t]+")
AR_EN_MIX_RE_1 = re.compile(rf"([{ARABIC_RANGES}])([A-Za-z0-9])")
AR_EN_MIX_RE_2 = re.compile(rf"([A-Za-z0-9])([{ARABIC_RANGES}])")

ALLOWED_LATIN_TOKENS = {
    "word",
    "powerpoint",
    "blackboard",
    "moodle",
    "zoom",
    "teams",
    "microsoft",
    "google",
    "drive",
    "calendar",
    "notion",
    "youtube",
    "gmail",
    "onedrive",
    "excel",
    "windows",
    "pdf",
    "docx",
    "txt",
    "power",
    "point",
    "googleclassroom",
    "classroom",
}

SUSPICIOUS_CHARS = set("ǚȦȧʫɒɓʺˀ˶ˬɐɔăíœð˅̷˝ĝżĊçëÿò§ˌˣˏ˟ʪʧʮ˄ΑЫа")

CHAR_REPLACEMENTS = {
    "ھ": "ه",
    "ة‌": "ة",
    "ﻻ": "لا",
    "ﻷ": "لأ",
    "ﻹ": "لإ",
    "ﻵ": "لآ",
    "أٔ": "أ",
    "إٔ": "إ",
    "أ": "أ",
    "ىٔ": "ئ",
    "ؤٔ": "ؤ",
    "ﻯ": "ى",
    "ﯨ": "ي",
    "ﯩ": "ي",
    "ﯪ": "ي",
    "ﯫ": "ي",
    "ﮫ": "ه",
    "ﮭ": "ه",
    "ﮮ": "ه",
    "ﮯ": "ه",
    "ئ": "ئ",
}

OPENAI_AVAILABLE = False
client = None
if USE_OPENAI_FOR_SUSPICIOUS_LINES and OPENAI_API_KEY:
    try:
        from openai import OpenAI

        client = OpenAI(api_key=OPENAI_API_KEY)
        OPENAI_AVAILABLE = True
    except Exception as error:
        print("OpenAI initialization failed:", error)


def norm_newlines(text):
    return text.replace("\r\n", "\n").replace("\r", "\n")


def is_arabic_char(char):
    return bool(ARABIC_CHAR_RE.match(char))


def arabic_char_ratio(text):
    chars = [char for char in text if not char.isspace()]
    if not chars:
        return 0.0
    return sum(is_arabic_char(char) for char in chars) / len(chars)


def tokenize(text):
    return WORD_RE.findall(text)


def token_has_suspicious_chars(token):
    return any(char in SUSPICIOUS_CHARS for char in token)


def token_is_likely_non_arabic_junk(token):
    token = token.strip()
    if not token:
        return False

    core = re.sub(r"^[^\w\u0600-\u06FF]+|[^\w\u0600-\u06FF]+$", "", token)
    if not core:
        return False

    lower = core.lower()
    if lower in ALLOWED_LATIN_TOKENS:
        return False

    has_arabic = bool(re.search(f"[{ARABIC_RANGES}]", core))
    has_english = bool(EN_CHAR_RE.search(core))

    if has_arabic and has_english:
        return True
    if token_has_suspicious_chars(core):
        return True
    if has_english and not has_arabic:
        return not bool(re.fullmatch(r"[A-Za-z]{1,4}", core))
    return False


def real_tokens(text):
    return [
        token
        for token in tokenize(text)
        if re.search(r"\w|[" + ARABIC_RANGES + "]", token)
    ]


def suspicious_token_ratio(text):
    tokens = real_tokens(text)
    if not tokens:
        return 0.0
    return sum(token_is_likely_non_arabic_junk(token) for token in tokens) / len(tokens)


def non_arabic_word_ratio(text):
    tokens = real_tokens(text)
    if not tokens:
        return 0.0

    non_arabic = 0
    for token in tokens:
        has_arabic = bool(re.search(f"[{ARABIC_RANGES}]", token))
        has_english = bool(EN_CHAR_RE.search(token))
        if has_english and not has_arabic and token.lower() not in ALLOWED_LATIN_TOKENS:
            non_arabic += 1
    return non_arabic / len(tokens)


def line_is_suspicious(line):
    line = line.strip()
    if len(line) < 20:
        return False
    return (
        arabic_char_ratio(line) < MAJOR_MIN_ARABIC_CHAR_RATIO
        or suspicious_token_ratio(line) > MAJOR_MAX_SUSPICIOUS_TOKEN_RATIO
        or non_arabic_word_ratio(line) > MAJOR_MAX_NON_ARABIC_WORD_RATIO
    )


def smart_arabic_spacing_fix(text, fix_log):
    replacements = (
        (
            AR_EN_MIX_RE_1,
            r"\1 \2",
            "Inserted spaces between Arabic and English/number sequences.",
        ),
        (
            AR_EN_MIX_RE_2,
            r"\1 \2",
            "Inserted spaces between English/number and Arabic sequences.",
        ),
        (SPACE_BEFORE_PUNCT_RE, r"\1", "Removed spaces before punctuation."),
        (SPACE_AFTER_OPEN_RE, r"\1", "Removed spaces after opening brackets."),
        (MULTISPACE_RE, " ", "Collapsed repeated spaces."),
    )
    for pattern, replacement, message in replacements:
        updated = pattern.sub(replacement, text)
        if updated != text:
            fix_log.append(message)
            text = updated
    return text


def safe_global_char_normalization(text, fix_log):
    updated = unicodedata.normalize("NFKC", text)
    if updated != text:
        fix_log.append("Applied Unicode NFKC normalization.")
        text = updated

    updated = INVISIBLES_RE.sub("", text)
    if updated != text:
        fix_log.append("Removed invisible and bidi control characters.")
        text = updated

    replacements = []
    for old, new in CHAR_REPLACEMENTS.items():
        count = text.count(old)
        if count:
            text = text.replace(old, new)
            replacements.append(f"{old}->{new} x{count}")
    if replacements:
        fix_log.append("Applied character replacements: " + "; ".join(replacements))

    text = norm_newlines(text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text)


def try_fix_common_mixed_words(text, fix_log):
    patterns = (
        (r"باستخدامPowerPoint", "باستخدام PowerPoint"),
        (r"باستخدامWord", "باستخدام Word"),
        (r"Google ?Drive", "Google Drive"),
        (r"Calendar ?Google", "Google Calendar"),
        (r"Teams ?Microsoft", "Microsoft Teams"),
        (r"Drive Google", "Google Drive"),
    )
    replacements = []
    for pattern, replacement in patterns:
        text, count = re.subn(pattern, replacement, text)
        if count:
            replacements.append(f"{pattern} -> {replacement} x{count}")
    if replacements:
        fix_log.append("Applied known mixed-word repairs: " + "; ".join(replacements))
    return text


def maybe_fix_single_corrupted_token(token):
    prefix = re.match(r"^[^\w\u0600-\u06FF]*", token).group(0)
    suffix = re.search(r"[^\w\u0600-\u06FF]*$", token).group(0)
    core_end = len(token) - len(suffix) if suffix else len(token)
    core = token[len(prefix) : core_end]
    if not core:
        return token, None

    arabic_chars = sum(is_arabic_char(char) for char in core)
    suspicious_chars = sum(char in SUSPICIOUS_CHARS for char in core)
    if arabic_chars >= 2 and suspicious_chars == 1:
        cleaned = "".join(char for char in core if char not in SUSPICIOUS_CHARS)
        if len(cleaned) >= max(2, len(core) - 2):
            fixed = prefix + cleaned + suffix
            return fixed, f"{token} -> {fixed}"

    has_arabic = bool(re.search(f"[{ARABIC_RANGES}]", core))
    has_english = bool(EN_CHAR_RE.search(core))
    if has_arabic and has_english:
        cleaned = re.sub(r"[A-Za-z]", "", core)
        if len(cleaned) >= 2 and re.search(f"[{ARABIC_RANGES}]", cleaned):
            fixed = prefix + cleaned + suffix
            return fixed, f"{token} -> {fixed}"

    return token, None


def token_level_minor_repairs(text, fix_log):
    parts = re.split(r"(\s+)", text)
    replacements = []
    for index, part in enumerate(parts):
        if not part or part.isspace():
            continue
        fixed, description = maybe_fix_single_corrupted_token(part)
        if description:
            parts[index] = fixed
            replacements.append(description)

    if replacements:
        preview = replacements[:30]
        message = "Token-level minor repairs:\n  - " + "\n  - ".join(preview)
        if len(replacements) > len(preview):
            message += f"\n  ... and {len(replacements) - len(preview)} more"
        fix_log.append(message)
    return "".join(parts)


def repair_line_with_openai(line):
    if not OPENAI_AVAILABLE:
        return line, None

    prompt = f"""
أصلح السطر التالي فقط إذا كان فيه فساد ترميزي أو أحرف خاطئة بسيطة.
- أعد كتابة السطر نفسه فقط.
- لا تختصر أو تضف شرحًا أو تعيد صياغة الأسلوب.
- لا تصحح المعنى إلا إذا كان الفساد الترميزي واضحًا جدًا.
- إذا كان السطر غير قابل للإصلاح بثقة، فأعده كما هو.

السطر:
{line}
""".strip()

    try:
        response = client.responses.create(model=OPENAI_MODEL, input=prompt)
        fixed = response.output_text.strip()
        return (fixed, "OpenAI repaired suspicious line.") if fixed else (line, None)
    except Exception as error:
        return line, f"OpenAI failed: {error}"


def classify_file(text):
    arabic_ratio = arabic_char_ratio(text)
    suspicious_ratio = suspicious_token_ratio(text)
    non_arabic_ratio = non_arabic_word_ratio(text)
    return {
        "arabic_char_ratio": round(arabic_ratio, 4),
        "suspicious_token_ratio": round(suspicious_ratio, 4),
        "non_arabic_word_ratio": round(non_arabic_ratio, 4),
        "is_largely_affected": (
            arabic_ratio < MAJOR_MIN_ARABIC_CHAR_RATIO
            or suspicious_ratio > MAJOR_MAX_SUSPICIOUS_TOKEN_RATIO
            or non_arabic_ratio > MAJOR_MAX_NON_ARABIC_WORD_RATIO
        ),
    }


def repair_text(text):
    original = text
    fix_log = []

    text = safe_global_char_normalization(text, fix_log)
    text = try_fix_common_mixed_words(text, fix_log)
    text = smart_arabic_spacing_fix(text, fix_log)
    text = token_level_minor_repairs(text, fix_log)

    if OPENAI_AVAILABLE:
        lines = text.split("\n")
        suspicious_indices = [
            index for index, line in enumerate(lines) if line_is_suspicious(line)
        ][:MAX_SUSPICIOUS_LINES_PER_FILE_FOR_OPENAI]
        line_fixes = []

        for index in suspicious_indices:
            old_line = lines[index]
            new_line, _ = repair_line_with_openai(old_line)
            if new_line != old_line:
                lines[index] = new_line
                line_fixes.append(
                    {
                        "line_no": index + 1,
                        "before": old_line[:300],
                        "after": new_line[:300],
                    }
                )

        if line_fixes:
            fix_log.append(
                "OpenAI suspicious-line repairs:\n  - "
                + "\n  - ".join(
                    f"line {item['line_no']}: {item['before']} -> {item['after']}"
                    for item in line_fixes[:20]
                )
            )
        text = "\n".join(lines)

    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"
    return text, text != original, fix_log


def write_csv(path, rows, fieldnames):
    with open(path, "w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


summary_rows = []
fix_rows = []
total_files = major_files = minor_processed = 0
changed_files = unchanged_files = failed_files = 0

for root, _, files in os.walk(INPUT_ROOT):
    for filename in files:
        if not filename.lower().endswith(".txt"):
            continue

        total_files += 1
        input_path = os.path.join(root, filename)
        relative_path = os.path.relpath(input_path, INPUT_ROOT)
        output_path = os.path.join(OUTPUT_ROOT, relative_path)
        major_copy_path = os.path.join(MAJOR_ROOT, relative_path)

        try:
            with open(input_path, "r", encoding="utf-8") as file:
                text = file.read()

            metrics = classify_file(text)
            row = {"file": relative_path, **metrics}

            if metrics["is_largely_affected"]:
                major_files += 1
                os.makedirs(os.path.dirname(major_copy_path), exist_ok=True)
                shutil.copy2(input_path, major_copy_path)
                row["status"] = "LARGELY_AFFECTED_REPORTED_ONLY"
                summary_rows.append(row)
                print(
                    f"[LARGELY AFFECTED] {relative_path}\n  Copied to: {major_copy_path}"
                )
                continue

            repaired_text, changed, fix_log = repair_text(text)
            minor_processed += 1
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            if OVERWRITE_OUTPUT or not os.path.exists(output_path):
                with open(output_path, "w", encoding="utf-8") as file:
                    file.write(repaired_text)

            row["status"] = "MINOR_FIXED" if changed else "MINOR_NO_CHANGE"
            summary_rows.append(row)
            if changed:
                changed_files += 1
                print(f"[FIXED] {relative_path}")
                for item in fix_log:
                    print(" - " + item.replace("\n", "\n   "))
                    fix_rows.append({"file": relative_path, "fix": item})
            else:
                unchanged_files += 1
                print(f"[NO CHANGE NEEDED] {relative_path}")

        except Exception as error:
            failed_files += 1
            summary_rows.append(
                {
                    "file": relative_path,
                    "arabic_char_ratio": None,
                    "suspicious_token_ratio": None,
                    "non_arabic_word_ratio": None,
                    "is_largely_affected": None,
                    "status": f"FAILED: {error}",
                }
            )
            print(f"[FAILED] {relative_path}\n  {error}")

summary_csv = os.path.join(REPORT_DIR, "repair_summary.csv")
fixes_csv = os.path.join(REPORT_DIR, "repair_fixes_log.csv")
major_csv = os.path.join(REPORT_DIR, "largely_affected_files.csv")
summary_fields = [
    "file",
    "arabic_char_ratio",
    "suspicious_token_ratio",
    "non_arabic_word_ratio",
    "is_largely_affected",
    "status",
]

write_csv(summary_csv, summary_rows, summary_fields)
write_csv(fixes_csv, fix_rows, ["file", "fix"])
write_csv(
    major_csv,
    [row for row in summary_rows if row["status"] == "LARGELY_AFFECTED_REPORTED_ONLY"],
    summary_fields,
)

print("\nDONE")
print(f"Total TXT files scanned: {total_files}")
print(f"Largely affected reported: {major_files}")
print(f"Minor-eligible processed: {minor_processed}")
print(f"Changed minor files: {changed_files}")
print(f"Unchanged minor files: {unchanged_files}")
print(f"Failed files: {failed_files}")
print("Reports:")
for path in (summary_csv, fixes_csv, major_csv):
    print(" -", path)

### Code block 5 — Use the OpenAI API to repair words split by spaces

This block targets Arabic words that were broken by stray spaces during extraction. It uses the OpenAI API to repair those cases while trying to preserve meaning and document structure.

In [ ]:
import os
import re
import time

from openai import OpenAI

INPUT_ROOT = os.environ["ARABIC_TEXT_INPUT_ROOT"]
OUTPUT_ROOT = os.environ["ARABIC_TEXT_OUTPUT_ROOT"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")

MAX_CHARS_PER_CHUNK = 12_000
SLEEP_BETWEEN_CALLS = 1.0
SKIP_EXISTING = True
PRINT_CHUNK_INFO = True

if os.path.abspath(INPUT_ROOT) == os.path.abspath(OUTPUT_ROOT):
    raise ValueError(
        "ARABIC_TEXT_INPUT_ROOT and ARABIC_TEXT_OUTPUT_ROOT must be different directories."
    )

client = OpenAI(api_key=OPENAI_API_KEY)


def normalize_newlines(text):
    return text.replace("\r\n", "\n").replace("\r", "\n")


def split_text_by_paragraphs(text, max_chars=MAX_CHARS_PER_CHUNK):
    """Split text by paragraphs, falling back to lines for oversized paragraphs."""
    text = normalize_newlines(text).strip()
    if not text:
        return []

    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        block = paragraph if not current else "\n\n" + paragraph
        if len(current) + len(block) <= max_chars:
            current += block
            continue

        if current:
            chunks.append(current)

        if len(paragraph) <= max_chars:
            current = paragraph
            continue

        current = ""
        for line in paragraph.split("\n"):
            piece = line if not current else "\n" + line
            if len(current) + len(piece) <= max_chars:
                current += piece
                continue

            if current:
                chunks.append(current)

            if len(line) <= max_chars:
                current = line
            else:
                for start in range(0, len(line), max_chars):
                    chunks.append(line[start : start + max_chars])
                current = ""

    if current:
        chunks.append(current)

    return chunks


def call_openai_fix(chunk_text):
    prompt = f"""
Fix only Arabic words that were incorrectly split by internal spaces.

Preserve wording, meaning, order, headings, bullets, numbering, punctuation,
and paragraph structure. Do not rewrite, summarize, or make grammar corrections
unless required to join a clearly split word. Preserve English text and formatting
unless they are clearly split incorrectly.

Return only the corrected text.

Text:
{chunk_text}
""".strip()

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )
    return response.output_text


def ensure_parent(path):
    os.makedirs(os.path.dirname(path), exist_ok=True)


def process_file(in_path, out_path):
    with open(in_path, "r", encoding="utf-8") as file:
        text = normalize_newlines(file.read())

    chunks = split_text_by_paragraphs(text)
    fixed_chunks = []

    print("=" * 80)
    print(f"FILE: {in_path}")
    print(f"CHUNKS: {len(chunks)}")

    for index, chunk in enumerate(chunks, start=1):
        if PRINT_CHUNK_INFO:
            preview = chunk[:160].replace("\n", " ")
            print(
                f"  Chunk {index}/{len(chunks)} | chars={len(chunk)} | preview={preview}"
            )

        fixed_chunks.append(call_openai_fix(chunk).strip())
        time.sleep(SLEEP_BETWEEN_CALLS)

    fixed_text = "\n\n".join(fixed_chunks).strip() + "\n"
    ensure_parent(out_path)

    with open(out_path, "w", encoding="utf-8") as file:
        file.write(fixed_text)

    print(f"WRITTEN: {out_path}")


total_files = 0
written_files = 0
failed_files = []

for root, _, files in os.walk(INPUT_ROOT):
    for filename in files:
        if not filename.lower().endswith(".txt"):
            continue

        total_files += 1
        in_path = os.path.join(root, filename)
        relative_path = os.path.relpath(in_path, INPUT_ROOT)
        out_path = os.path.join(OUTPUT_ROOT, relative_path)

        if SKIP_EXISTING and os.path.exists(out_path):
            print(f"SKIPPED (exists): {out_path}")
            continue

        try:
            process_file(in_path, out_path)
            written_files += 1
        except Exception as error:
            failed_files.append((in_path, str(error)))
            print(f"FAILED: {in_path}\n  Error: {error}")

print("\n" + "=" * 80)
print("DONE")
print(f"Total TXT files found: {total_files}")
print(f"Files written: {written_files}")
print(f"Failed files: {len(failed_files)}")

if failed_files:
    print("\nFAILED FILES:")
    for path, error in failed_files:
        print(f"- {path}\n  {error}")

### Code block 6 — Run a broader Arabic TXT cleaning pass

This block applies a larger cleaning pass that standardizes line structure, preserves headings and bullets, and keeps the cleaned Arabic TXT files ready for later filtering or analysis.

In [ ]:
!pip -q install --upgrade openai tqdm

import json
import os
import re
import shutil
from collections import defaultdict
from pathlib import Path

from google.colab import drive
from openai import OpenAI
from tqdm.auto import tqdm

drive.mount('/content/drive')

# Configure private input and output locations through environment variables.
DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "data"))
TXT_ROOT = Path(os.environ.get("TXT_ROOT", DRIVE_ROOT / "arabic_txt_input"))
PDF_ROOT = Path(os.environ.get("PDF_ROOT", DRIVE_ROOT / "arabic_pdf_input"))
CLEAN_ROOT = Path(os.environ.get("CLEAN_ROOT", DRIVE_ROOT / "arabic_txt_cleaned"))
INSPECTION_ROOT = Path(os.environ.get("INSPECTION_ROOT", DRIVE_ROOT / "arabic_txt_inspection"))
REPORT_JSON = Path(os.environ.get("REPORT_JSON", CLEAN_ROOT / "cleaning_report.json"))
REPORT_TXT = Path(os.environ.get("REPORT_TXT", CLEAN_ROOT / "cleaning_report.txt"))

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.4")
OPENAI_SAMPLE_CHARS = int(os.environ.get("OPENAI_SAMPLE_CHARS", "7000"))
CHECK_EVERY_FILE_WITH_OPENAI = os.environ.get(
    "CHECK_EVERY_FILE_WITH_OPENAI", "true"
).lower() in {"1", "true", "yes"}

assert TXT_ROOT.exists(), f"TXT root not found: {TXT_ROOT}"
assert PDF_ROOT.exists(), f"PDF root not found: {PDF_ROOT}"

CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
INSPECTION_ROOT.mkdir(parents=True, exist_ok=True)

if not OPENAI_API_KEY.strip():
    raise ValueError("Set OPENAI_API_KEY in the notebook environment.")

client = OpenAI(api_key=OPENAI_API_KEY)

ARABIC_RANGE_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]")
LATIN_RE = re.compile(r"[A-Za-z]")
DIGIT_RE = re.compile(r"[0-9٠-٩]")
WEIRD_SYMBOL_RUN_RE = re.compile(
    r"[^\w\s\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF.,;:!?؟،()\[\]{}\"'\-–—_*•●▪◦·/%+&=@#]+"
)
MULTISPACE_RE = re.compile(r"[ \t]+")

BULLET_LINE_RE = re.compile(
    r"^\s*("
    r"[-*•●▪◦‣–—]"
    r"|(?:\(?\d{1,3}\)?[\.\-:])"
    r"|(?:\(?[٠-٩]{1,3}\)?[\.\-:])"
    r"|(?:\(?[A-Za-z]\)?[\.\-:])"
    r"|(?:\(?[أ-ي]\)?[\.\-:])"
    r")\s+"
)
INLINE_BULLET_RE = re.compile(
    r"(?<!\n)([^\n])\s+(?=("
    r"[-*•●▪◦‣–—]\s+"
    r"|(?:\(?\d{1,3}\)?[\.\-:]\s+)"
    r"|(?:\(?[٠-٩]{1,3}\)?[\.\-:]\s+)"
    r"|(?:\(?[A-Za-z]\)?[\.\-:]\s+)"
    r"|(?:\(?[أ-ي]\)?[\.\-:]\s+)"
    r"))"
)
END_PUNCT_RE = re.compile(r"[\.!?؟؛:]\s*$")


def arabic_ratio(text):
    chars = [char for char in text if not char.isspace()]
    if not chars:
        return 0.0
    return sum(bool(ARABIC_RANGE_RE.search(char)) for char in chars) / len(chars)


def contains_arabic(text):
    return bool(ARABIC_RANGE_RE.search(text))


def is_bullet_line(line):
    return bool(BULLET_LINE_RE.match(line.strip()))


def looks_like_heading(line):
    line = line.strip()
    if not line or is_bullet_line(line) or len(line) > 90:
        return False
    if line.endswith((":", "：", "؛")):
        return True
    words = line.split()
    return (
        1 <= len(words) <= 12
        and not END_PUNCT_RE.search(line)
        and arabic_ratio(line) >= 0.5
    )


def normalize_basic(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00A0", " ").replace("\t", " ")
    return MULTISPACE_RE.sub(" ", re.sub(r"\n{3,}", "\n\n", text))


def insert_newlines_before_inline_bullets(text):
    count = 0
    previous = None
    while previous != text:
        previous = text
        text, replacements = INLINE_BULLET_RE.subn(r"\1\n", text)
        count += replacements
    return text, count


def swap_parentheses_for_arabic_segments(text):
    """Reverse parenthesis direction only for Arabic-majority parenthetical text."""
    swaps = 0
    pattern = re.compile(r"\(([^()\n]{1,300})\)")

    def replace(match):
        nonlocal swaps
        inner = match.group(1).strip()
        if not inner:
            return match.group(0)
        if arabic_ratio(inner) >= 0.45 and len(LATIN_RE.findall(inner)) <= max(2, len(inner) * 0.08):
            swaps += 1
            return f"){inner}("
        return match.group(0)

    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(replace, text)
    return text, swaps


def rebuild_text_structure(text):
    """Join extraction-wrapped lines while preserving headings, lists, and sentence boundaries."""
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    output, paragraph = [], []
    joined_count = 0

    def flush():
        if paragraph:
            output.append(" ".join(paragraph).strip())
            paragraph.clear()

    for line in lines:
        if is_bullet_line(line) or looks_like_heading(line):
            flush()
            output.append(line)
            continue
        paragraph.append(line)
        if END_PUNCT_RE.search(line):
            flush()
        else:
            joined_count += 1

    flush()
    cleaned = "\n".join(output).strip()
    return re.sub(r"\n{3,}", "\n\n", cleaned), joined_count


def suspicious_heuristics(text):
    reasons, weird_lines = [], []
    weird_symbol_hits = low_arabic_hits = fractured_token_hits = 0

    for line in (line.strip() for line in text.splitlines() if line.strip()):
        if len(weird_lines) >= 8 and weird_symbol_hits >= 5:
            break
        arabic = arabic_ratio(line)
        latin = len(LATIN_RE.findall(line))
        if WEIRD_SYMBOL_RUN_RE.search(line):
            weird_symbol_hits += 1
        if arabic < 0.25 and contains_arabic(line) and latin < 3 and len(line) > 8:
            low_arabic_hits += 1
            weird_lines.append(line[:200])
        tokens = line.split()
        if len(tokens) >= 6:
            one_char_arabic = sum(len(token) == 1 and contains_arabic(token) for token in tokens)
            if one_char_arabic >= max(3, len(tokens) // 3):
                fractured_token_hits += 1
                weird_lines.append(line[:200])

    if weird_symbol_hits >= 5:
        reasons.append(f"many weird-symbol lines ({weird_symbol_hits})")
    if low_arabic_hits >= 4:
        reasons.append(f"many suspicious non-Arabic-looking Arabic lines ({low_arabic_hits})")
    if fractured_token_hits >= 3:
        reasons.append(f"possible reversed/fractured Arabic tokens ({fractured_token_hits})")

    return {
        "suspicious": bool(reasons),
        "reasons": reasons,
        "sample_weird_lines": weird_lines[:8],
    }


def detect_corruption_with_openai(text, file_rel_path, heuristic_info):
    prompt = f"""
Check this extracted Arabic TXT file for severe OCR or extraction corruption requiring manual inspection.

Flag corruption only for reversed or fractured words, gibberish, severe ordering failures, many unusual symbols, or unreadable passages. Do not flag ordinary punctuation, occasional English terms, filenames, common spacing issues, or isolated minor OCR errors.

File: {file_rel_path}
Local heuristic results: {json.dumps(heuristic_info, ensure_ascii=False)}

Return only valid JSON:
{{"is_corrupted": true, "severity": "none|minor|moderate|severe", "confidence": 0.0, "reason": "short reason", "evidence": ["short evidence"]}}

Text sample:
{text[:OPENAI_SAMPLE_CHARS]}
""".strip()

    response = client.responses.create(model=OPENAI_MODEL, input=prompt)
    raw = response.output_text.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, flags=re.S)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass

    suspicious = heuristic_info["suspicious"]
    return {
        "is_corrupted": suspicious,
        "severity": "moderate" if suspicious else "none",
        "confidence": 0.35,
        "reason": "Could not parse model JSON response; local fallback used.",
        "evidence": heuristic_info["sample_weird_lines"][:2],
    }


def build_pdf_index(pdf_root):
    exact_rel, by_stem = {}, defaultdict(list)
    for pdf_file in pdf_root.rglob("*.pdf"):
        exact_rel[pdf_file.relative_to(pdf_root).as_posix()] = pdf_file
        by_stem[pdf_file.stem].append(pdf_file)
    return exact_rel, by_stem


def choose_matching_pdf(txt_rel_path, txt_file, exact_pdf_rel, pdf_by_stem):
    expected = txt_rel_path.with_suffix(".pdf").as_posix()
    if expected in exact_pdf_rel:
        return exact_pdf_rel[expected], "same_relative_path"

    candidates = pdf_by_stem.get(txt_file.stem, [])
    if not candidates:
        return None, "no_stem_match"

    parent = txt_rel_path.parent.as_posix()
    parent_matches = [pdf for pdf in candidates if pdf.parent.as_posix().endswith(parent)]
    if len(parent_matches) == 1:
        return parent_matches[0], "same_stem_similar_parent"
    if len(candidates) == 1:
        return candidates[0], "unique_stem_match"
    return candidates[0], "ambiguous_first_stem_match"


print("Indexing source PDFs...")
exact_pdf_rel, pdf_by_stem = build_pdf_index(PDF_ROOT)
all_txt_files = sorted(TXT_ROOT.rglob("*.txt"))
print(f"Found {len(all_txt_files)} TXT files.")

report = []
for txt_file in tqdm(all_txt_files, desc="Processing TXT files"):
    rel_path = txt_file.relative_to(TXT_ROOT)
    clean_out = CLEAN_ROOT / rel_path
    inspect_txt_out = INSPECTION_ROOT / rel_path
    inspect_pdf_out = INSPECTION_ROOT / rel_path.with_suffix(".pdf")
    clean_out.parent.mkdir(parents=True, exist_ok=True)
    inspect_txt_out.parent.mkdir(parents=True, exist_ok=True)

    try:
        raw = txt_file.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        raw = txt_file.read_text(encoding="utf-8-sig", errors="ignore")

    text = normalize_basic(raw)
    text, inline_bullet_breaks = insert_newlines_before_inline_bullets(text)
    text, paren_swaps = swap_parentheses_for_arabic_segments(text)
    cleaned, joined_line_count = rebuild_text_structure(text)
    cleaned = re.sub(r"[ ]+\n", "\n", cleaned)
    cleaned = re.sub(r"\n[ ]+", "\n", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip() + "\n"
    clean_out.write_text(cleaned, encoding="utf-8")

    heuristic_info = suspicious_heuristics(cleaned)
    openai_checked = CHECK_EVERY_FILE_WITH_OPENAI or heuristic_info["suspicious"]
    ai_result = (
        detect_corruption_with_openai(cleaned, rel_path.as_posix(), heuristic_info)
        if openai_checked
        else {
            "is_corrupted": False,
            "severity": "none",
            "confidence": 0.0,
            "reason": "Skipped model check because local heuristics found no issue.",
            "evidence": [],
        }
    )

    flagged = bool(ai_result.get("is_corrupted", False))
    pdf_match, pdf_match_mode = choose_matching_pdf(rel_path, txt_file, exact_pdf_rel, pdf_by_stem)
    copied_pdf = False
    if flagged:
        shutil.copy2(txt_file, inspect_txt_out)
        if pdf_match and pdf_match.exists():
            inspect_pdf_out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(pdf_match, inspect_pdf_out)
            copied_pdf = True

    report.append({
        "txt_file": str(txt_file),
        "relative_path": rel_path.as_posix(),
        "cleaned_output": str(clean_out),
        "original_chars": len(raw),
        "cleaned_chars": len(cleaned),
        "inline_bullet_breaks_added": inline_bullet_breaks,
        "joined_line_count_estimate": joined_line_count,
        "arabic_parenthesis_swaps": paren_swaps,
        "heuristic_suspicious": heuristic_info["suspicious"],
        "heuristic_reasons": heuristic_info["reasons"],
        "openai_checked": openai_checked,
        "openai_result": ai_result,
        "flagged_for_inspection": flagged,
        "copied_flagged_txt_to_inspection": flagged,
        "matching_pdf_found": pdf_match is not None,
        "matching_pdf_path": str(pdf_match) if pdf_match else None,
        "matching_pdf_mode": pdf_match_mode,
        "copied_matching_pdf_to_inspection": copied_pdf,
    })

REPORT_JSON.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

flagged_items = [item for item in report if item["flagged_for_inspection"]]
with REPORT_TXT.open("w", encoding="utf-8") as report_file:
    report_file.write("Arabic TXT Cleaning and Corruption Detection Report\n")
    report_file.write("=" * 70 + "\n\n")
    report_file.write(f"TXT root: {TXT_ROOT}\nPDF root: {PDF_ROOT}\n")
    report_file.write(f"Clean output root: {CLEAN_ROOT}\nInspection root: {INSPECTION_ROOT}\n\n")
    report_file.write(f"Total TXT files processed: {len(report)}\n")
    report_file.write(f"Flagged for inspection: {len(flagged_items)}\n\n")
    report_file.write("Flagged files:\n" + "-" * 70 + "\n")
    for item in flagged_items:
        result = item["openai_result"]
        report_file.write(
            f"{item['relative_path']}\n"
            f"  Reason: {result.get('reason')}\n"
            f"  Severity: {result.get('severity')}\n"
            f"  Confidence: {result.get('confidence')}\n"
            f"  Matching PDF found: {item['matching_pdf_found']}\n"
            f"  Matching PDF path: {item['matching_pdf_path']}\n\n"
        )

print(f"Cleaned TXT files: {CLEAN_ROOT}")
print(f"Inspection copies: {INSPECTION_ROOT}")
print(f"JSON report: {REPORT_JSON}")
print(f"Text report: {REPORT_TXT}")

### Code block 7 — Convert source PDFs into scanned-image PDFs

This block rebuilds each PDF as an image-based scanned PDF. This is useful for problematic Arabic PDFs when a direct text-extraction path is unreliable.

In [ ]:
!pip -q install --upgrade openai requests tqdm pillow

import base64
import io
import json
import mimetypes
import os
import re
import shutil
import time
import zipfile
from pathlib import Path

import requests
from google.colab import drive
from openai import OpenAI
from PIL import Image
from tqdm.auto import tqdm

# Mount Google Drive when running in Colab.
drive.mount("/content/drive")

# Configuration is supplied through environment variables rather than notebook secrets.
DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "data"))
SOURCE_PDF_ROOT = Path(os.environ["SOURCE_PDF_ROOT"])
SCANNED_PDF_ROOT = Path(os.environ.get("SCANNED_PDF_ROOT", DRIVE_ROOT / "scanned_pdfs"))
TXT_OUTPUT_ROOT = Path(os.environ.get("TXT_OUTPUT_ROOT", DRIVE_ROOT / "ocr_text"))
TEMP_ROOT = Path(os.environ.get("TEMP_ROOT", "/content/pdf_ocr_pipeline"))
SUMMARY_PATH = Path(os.environ.get("SUMMARY_PATH", DRIVE_ROOT / "pdf_ocr_summary.json"))

ILOVE_PUBLIC_KEY = os.environ["ILOVE_PUBLIC_KEY"]
ILOVE_SECRET_KEY = os.environ["ILOVE_SECRET_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
JPEG_QUALITY = 92
OCR_MAX_IMAGE_DIM = 1800
SLEEP_BETWEEN_OPENAI_CALLS = 0.4
OPENAI_MAX_RETRIES = 5
ILOVE_API_BASE = "https://api.ilovepdf.com/v1"
ILOVE_REGION = "us"
SKIP_ALREADY_DONE = True

assert SOURCE_PDF_ROOT.exists(), f"Source root not found: {SOURCE_PDF_ROOT}"

for path in (SCANNED_PDF_ROOT, TXT_OUTPUT_ROOT, TEMP_ROOT):
    path.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENAI_API_KEY)


def safe_mkdir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def rel_from_source(pdf_path: Path) -> Path:
    return pdf_path.relative_to(SOURCE_PDF_ROOT)


def normalize_text_basic(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00A0", " ").replace("\ufeff", "")
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def is_arabic_char(char: str) -> bool:
    return (
        "\u0600" <= char <= "\u06FF"
        or "\u0750" <= char <= "\u077F"
        or "\u08A0" <= char <= "\u08FF"
        or "\uFB50" <= char <= "\uFDFF"
        or "\uFE70" <= char <= "\uFEFF"
    )


def arabic_ratio(text: str) -> float:
    characters = [char for char in text if not char.isspace()]
    if not characters:
        return 0.0
    return sum(is_arabic_char(char) for char in characters) / len(characters)


def looks_like_bullet(line: str) -> bool:
    return bool(
        re.match(
            r"^("
            r"[-*•●▪◦‣–—]\s+"
            r"|(?:\(?\d{1,3}\)?[.\-:])\s+"
            r"|(?:\(?[٠-٩]{1,3}\)?[.\-:])\s+"
            r"|(?:\(?[A-Za-z]\)?[.\-:])\s+"
            r"|(?:\(?[أ-ي]\)?[.\-:])\s+"
            r")",
            line.strip(),
        )
    )


def looks_like_heading(line: str) -> bool:
    line = line.strip()
    if not line or len(line) > 100 or looks_like_bullet(line):
        return False
    if line.endswith((":", "：", "؛")):
        return True

    words = line.split()
    return (
        1 <= len(words) <= 12
        and not re.search(r"[.!?؟]$", line)
        and arabic_ratio(line) >= 0.45
    )


def should_preserve_newline_between(previous: str, current: str) -> bool:
    previous = previous.strip()
    current = current.strip()
    if not previous or not current:
        return True
    if looks_like_bullet(previous) or looks_like_bullet(current):
        return True
    if looks_like_heading(previous):
        return True
    return bool(re.search(r"[.!?؟:؛]\s*$", previous))


def insert_newline_before_inline_bullets(text: str) -> str:
    pattern = re.compile(
        r"(?<!\n)([^\n])\s+(?=("
        r"[-*•●▪◦‣–—]\s+"
        r"|(?:\(?\d{1,3}\)?[.\-:]\s+)"
        r"|(?:\(?[٠-٩]{1,3}\)?[.\-:]\s+)"
        r"|(?:\(?[A-Za-z]\)?[.\-:]\s+)"
        r"|(?:\(?[أ-ي]\)?[.\-:]\s+)"
        r"))"
    )
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(r"\1\n", text)
    return text


def merge_soft_linebreaks(text: str) -> str:
    # Preserve structural breaks while joining lines wrapped only by page width.
    lines = [line.strip() for line in insert_newline_before_inline_bullets(normalize_text_basic(text)).split("\n")]
    output = []
    buffer = []

    def flush_buffer():
        nonlocal buffer
        if buffer:
            output.append(" ".join(buffer).strip())
            buffer = []

    for line in lines:
        if not line:
            flush_buffer()
        elif looks_like_bullet(line) or looks_like_heading(line):
            flush_buffer()
            output.append(line)
        elif not buffer:
            buffer = [line]
        elif should_preserve_newline_between(buffer[-1], line):
            flush_buffer()
            buffer = [line]
        else:
            buffer.append(line)

    flush_buffer()
    return re.sub(r"\n{3,}", "\n\n", "\n".join(output)).strip() + "\n"


def image_to_base64_data_url(image_path: Path) -> str:
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        width, height = image.size
        scale = min(1.0, OCR_MAX_IMAGE_DIM / max(width, height))
        if scale < 1.0:
            image = image.resize(
                (int(width * scale), int(height * scale)),
                Image.LANCZOS,
            )
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=JPEG_QUALITY, optimize=True)

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


def natural_key(value):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", str(value))]


def ilove_auth_token() -> str:
    response = requests.post(
        f"{ILOVE_API_BASE}/auth",
        auth=(ILOVE_PUBLIC_KEY, ILOVE_SECRET_KEY),
        data={"public_key": ILOVE_PUBLIC_KEY},
        timeout=120,
    )
    response.raise_for_status()
    token = response.json().get("token")
    if not token:
        raise RuntimeError("iLovePDF authentication did not return a token.")
    return token


def ilove_headers(token: str) -> dict:
    return {"Authorization": f"Bearer {token}"}


def ilove_start_task(tool: str, token: str) -> dict:
    response = requests.get(
        f"{ILOVE_API_BASE}/start/{tool}/{ILOVE_REGION}",
        headers=ilove_headers(token),
        timeout=120,
    )
    response.raise_for_status()
    return response.json()


def ilove_upload_file(server: str, task_id: str, local_path: Path, token: str) -> dict:
    with local_path.open("rb") as file_handle:
        response = requests.post(
            f"https://{server}/v1/upload",
            headers=ilove_headers(token),
            data={"task": task_id},
            files={
                "file": (
                    local_path.name,
                    file_handle,
                    mimetypes.guess_type(str(local_path))[0] or "application/octet-stream",
                )
            },
            timeout=600,
        )
    response.raise_for_status()
    return response.json()


def ilove_process(
    server: str,
    task_id: str,
    tool: str,
    files_data: list,
    token: str,
    extra_params: dict | None = None,
) -> dict:
    payload = {"task": task_id, "tool": tool, "files": files_data}
    if extra_params:
        payload.update(extra_params)

    response = requests.post(
        f"https://{server}/v1/process",
        headers={**ilove_headers(token), "Content-Type": "application/json"},
        json=payload,
        timeout=1200,
    )
    response.raise_for_status()
    return response.json()


def ilove_download(server: str, task_id: str, output_path: Path, token: str):
    with requests.get(
        f"https://{server}/v1/download/{task_id}",
        headers=ilove_headers(token),
        stream=True,
        timeout=1200,
    ) as response:
        response.raise_for_status()
        with output_path.open("wb") as file_handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file_handle.write(chunk)


def pdf_to_jpg_zip_ilovepdf(pdf_path: Path, output_zip_path: Path):
    token = ilove_auth_token()
    task = ilove_start_task("pdfjpg", token)
    uploaded = ilove_upload_file(task["server"], task["task"], pdf_path, token)
    files_data = [{
        "server_filename": uploaded["server_filename"],
        "filename": uploaded.get("filename", pdf_path.name),
    }]
    ilove_process(
        task["server"],
        task["task"],
        "pdfjpg",
        files_data,
        token,
        {"pdfjpg_mode": "pages"},
    )
    ilove_download(task["server"], task["task"], output_zip_path, token)


def jpgs_to_pdf_ilovepdf(jpg_paths: list[Path], output_pdf_path: Path):
    token = ilove_auth_token()
    task = ilove_start_task("imagepdf", token)
    files_data = []

    for image_path in jpg_paths:
        uploaded = ilove_upload_file(task["server"], task["task"], image_path, token)
        files_data.append({
            "server_filename": uploaded["server_filename"],
            "filename": uploaded.get("filename", image_path.name),
        })

    ilove_process(
        task["server"],
        task["task"],
        "imagepdf",
        files_data,
        token,
        {
            "pagesize": "fit",
            "margin": 0,
            "orientation": "portrait",
            "merge_after": True,
        },
    )
    ilove_download(task["server"], task["task"], output_pdf_path, token)


def make_scanned_pdf_from_pdf(pdf_path: Path, scanned_pdf_path: Path, temp_dir: Path) -> list[Path]:
    safe_mkdir(temp_dir)
    zip_path = temp_dir / f"{pdf_path.stem}_pages.zip"
    image_dir = temp_dir / f"{pdf_path.stem}_jpg_pages"
    safe_mkdir(image_dir)

    pdf_to_jpg_zip_ilovepdf(pdf_path, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(image_dir)

    jpg_paths = sorted(
        (path for path in image_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg"}),
        key=lambda path: natural_key(path.name),
    )
    if not jpg_paths:
        raise RuntimeError(f"No JPG pages extracted for {pdf_path}")

    safe_mkdir(scanned_pdf_path.parent)
    jpgs_to_pdf_ilovepdf(jpg_paths, scanned_pdf_path)
    return jpg_paths


def openai_ocr_image(image_path: Path) -> str:
    prompt = (
        "Transcribe this page exactly into plain text.\n\n"
        "Preserve Arabic, English, numbers, symbols, and visible formulas. Do not "
        "summarize, explain, or correct wording. Keep headings and bullet points. "
        "Remove line breaks caused only by printed line width, but preserve paragraph, "
        "heading, and list-item breaks. Return only the page text."
    )
    data_url = image_to_base64_data_url(image_path)
    last_error = None

    for attempt in range(1, OPENAI_MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=OPENAI_MODEL,
                input=[{
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": prompt},
                        {"type": "input_image", "image_url": data_url},
                    ],
                }],
            )
            return response.output_text.strip()
        except Exception as error:
            last_error = error
            print(f"OpenAI OCR retry {attempt}/{OPENAI_MAX_RETRIES} for {image_path.name}: {error}")
            time.sleep(min(20, 2 ** (attempt - 1)))

    raise RuntimeError(f"OpenAI OCR failed for {image_path}: {last_error}")


def ocr_jpg_pages_to_txt(jpg_paths: list[Path], output_path: Path):
    page_texts = []
    for image_path in tqdm(jpg_paths, desc=f"OCR {output_path.name}", leave=False):
        text = merge_soft_linebreaks(normalize_text_basic(openai_ocr_image(image_path)))
        page_texts.append(text.strip())
        time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)

    safe_mkdir(output_path.parent)
    output_path.write_text("\n\n".join(text for text in page_texts if text).strip() + "\n", encoding="utf-8")


pdf_files = sorted(SOURCE_PDF_ROOT.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files under: {SOURCE_PDF_ROOT}")

summary = {"processed": [], "skipped": [], "failed": []}

for pdf_path in pdf_files:
    relative_path = rel_from_source(pdf_path)
    scanned_pdf_path = (SCANNED_PDF_ROOT / relative_path).with_suffix(".pdf")
    txt_output_path = (TXT_OUTPUT_ROOT / relative_path).with_suffix(".txt")

    if SKIP_ALREADY_DONE and scanned_pdf_path.exists() and txt_output_path.exists():
        print(f"SKIP (already done): {relative_path}")
        summary["skipped"].append(str(relative_path))
        continue

    print("=" * 100)
    print(f"PROCESSING: {relative_path}")

    temp_dir = TEMP_ROOT / relative_path.parent / pdf_path.stem
    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)
    safe_mkdir(temp_dir)

    try:
        page_jpgs = make_scanned_pdf_from_pdf(pdf_path, scanned_pdf_path, temp_dir)
        print(f"Scanned PDF saved -> {scanned_pdf_path}")

        ocr_jpg_pages_to_txt(page_jpgs, txt_output_path)
        print(f"TXT saved -> {txt_output_path}")

        summary["processed"].append({
            "relative_path": str(relative_path),
            "source_pdf": str(pdf_path),
            "scanned_pdf": str(scanned_pdf_path),
            "txt_output": str(txt_output_path),
            "page_count": len(page_jpgs),
        })
    except Exception as error:
        print(f"FAILED: {relative_path} -> {error}")
        summary["failed"].append({"relative_path": str(relative_path), "error": str(error)})

safe_mkdir(SUMMARY_PATH.parent)
with SUMMARY_PATH.open("w", encoding="utf-8") as file_handle:
    json.dump(summary, file_handle, ensure_ascii=False, indent=2)

print("\nDONE")
print(f"Scanned PDFs root: {SCANNED_PDF_ROOT}")
print(f"TXT output root:   {TXT_OUTPUT_ROOT}")
print(f"Summary JSON:      {SUMMARY_PATH}")
print(f"Processed: {len(summary['processed'])}")
print(f"Skipped:   {len(summary['skipped'])}")
print(f"Failed:    {len(summary['failed'])}")

### Code block 8 — Resume the local scanned-PDF pipeline without iLovePDF

This block is the resume-safe version of the scanned-PDF workflow. It skips already-finished files and continues the conversion process locally when the external conversion path is unavailable or unreliable.

In [ ]:
!pip -q install --upgrade "requests==2.32.4" "pillow<12" "openai>=1.0.0" "pymupdf>=1.24.0" "reportlab>=4.0.0" tqdm

from google.colab import drive
drive.mount("/content/drive")

import base64
import gc
import io
import json
import os
import re
import shutil
import time
from pathlib import Path

import fitz
from openai import OpenAI
from PIL import Image
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas
from tqdm.auto import tqdm

# Configure these paths for the mounted drive or another accessible storage location.
DRIVE_ROOT = Path(os.getenv("DRIVE_ROOT", "data"))
SOURCE_PDF_ROOT = Path(os.getenv("SOURCE_PDF_ROOT", str(DRIVE_ROOT / "input_pdfs")))
SCANNED_PDF_ROOT = Path(os.getenv("SCANNED_PDF_ROOT", str(DRIVE_ROOT / "scanned_pdfs")))
TXT_OUTPUT_ROOT = Path(os.getenv("TXT_OUTPUT_ROOT", str(DRIVE_ROOT / "ocr_text")))
TEMP_ROOT = Path(os.getenv("TEMP_ROOT", str(DRIVE_ROOT / "_tmp_pdf_ocr")))
REPORT_PATH = Path(os.getenv("REPORT_PATH", str(DRIVE_ROOT / "pdf_ocr_report.json")))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

SKIP_IF_TXT_EXISTS = True
SKIP_IF_SCANNED_AND_TXT_EXIST = True
REBUILD_SCANNED_IF_MISSING = True
REUSE_EXISTING_SCANNED_FOR_OCR = True
OVERWRITE_EXISTING_TXT = False

RENDER_DPI_FOR_SCANNED = 170
JPEG_QUALITY = 85
OCR_MAX_IMAGE_DIM = 1800
OPENAI_MAX_RETRIES = 5
SLEEP_BETWEEN_OPENAI_CALLS = 0.4
CLEAN_TEMP_AFTER_EACH_FILE = True
PROCESS_ONLY_EXT = ".pdf"

assert SOURCE_PDF_ROOT.exists(), f"Missing source folder: {SOURCE_PDF_ROOT}"
assert OPENAI_API_KEY, "Set the OPENAI_API_KEY environment variable."

SCANNED_PDF_ROOT.mkdir(parents=True, exist_ok=True)
TXT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENAI_API_KEY)


def normalize_text_basic(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00A0", " ").replace("\ufeff", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def is_arabic_char(ch: str) -> bool:
    return (
        "\u0600" <= ch <= "\u06FF"
        or "\u0750" <= ch <= "\u077F"
        or "\u08A0" <= ch <= "\u08FF"
        or "\uFB50" <= ch <= "\uFDFF"
        or "\uFE70" <= ch <= "\uFEFF"
    )


def arabic_ratio(text: str) -> float:
    chars = [char for char in text if not char.isspace()]
    if not chars:
        return 0.0
    return sum(is_arabic_char(char) for char in chars) / len(chars)


def looks_like_bullet(line: str) -> bool:
    return bool(
        re.match(
            r"^("
            r"[-*•●▪◦‣–—]\s+"
            r"|(?:\(?\d{1,3}\)?[\.\-:])\s+"
            r"|(?:\(?[٠-٩]{1,3}\)?[\.\-:])\s+"
            r"|(?:\(?[A-Za-z]\)?[\.\-:])\s+"
            r"|(?:\(?[أ-ي]\)?[\.\-:])\s+"
            r")",
            line.strip(),
        )
    )


def looks_like_heading(line: str) -> bool:
    text = line.strip()
    if not text or len(text) > 100 or looks_like_bullet(text):
        return False
    if text.endswith((":", "؛", "：")):
        return True
    words = text.split()
    return (
        1 <= len(words) <= 12
        and not re.search(r"[.!?؟]$", text)
        and arabic_ratio(text) >= 0.45
    )


def should_preserve_newline_between(previous_line: str, next_line: str) -> bool:
    previous = previous_line.strip()
    following = next_line.strip()
    if not previous or not following:
        return True
    if looks_like_bullet(previous) or looks_like_bullet(following):
        return True
    if looks_like_heading(previous):
        return True
    return bool(re.search(r"[.!?؟:؛]\s*$", previous))


def insert_newline_before_inline_bullets(text: str) -> str:
    pattern = re.compile(
        r"(?<!\n)([^\n])\s+(?=("
        r"[-*•●▪◦‣–—]\s+"
        r"|(?:\(?\d{1,3}\)?[\.\-:]\s+)"
        r"|(?:\(?[٠-٩]{1,3}\)?[\.\-:]\s+)"
        r"|(?:\(?[A-Za-z]\)?[\.\-:]\s+)"
        r"|(?:\(?[أ-ي]\)?[\.\-:]\s+)"
        r"))"
    )
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(r"\1\n", text)
    return text


def merge_soft_linebreaks(text: str) -> str:
    text = insert_newline_before_inline_bullets(normalize_text_basic(text))
    lines = [line.strip() for line in text.split("\n")]
    output = []
    buffer = []

    def flush_buffer() -> None:
        nonlocal buffer
        if buffer:
            output.append(" ".join(buffer).strip())
            buffer = []

    for line in lines:
        if not line:
            flush_buffer()
        elif looks_like_bullet(line) or looks_like_heading(line):
            flush_buffer()
            output.append(line)
        elif not buffer:
            buffer = [line]
        elif should_preserve_newline_between(buffer[-1], line):
            flush_buffer()
            buffer = [line]
        else:
            buffer.append(line)

    flush_buffer()
    return re.sub(r"\n{3,}", "\n\n", "\n".join(item for item in output if item.strip())).strip() + "\n"


def render_pdf_to_jpgs(
    pdf_path: Path,
    output_dir: Path,
    dpi: int = RENDER_DPI_FOR_SCANNED,
    jpeg_quality: int = JPEG_QUALITY,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    page_paths = []

    with fitz.open(pdf_path) as document:
        for index in range(len(document)):
            page = document.load_page(index)
            pixmap = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = output_dir / f"page_{index + 1:04d}.jpg"
            pixmap.save(str(image_path), jpg_quality=jpeg_quality)
            page_paths.append(image_path)
            del pixmap
            del page
            gc.collect()

    return page_paths


def build_pdf_from_images(image_paths: list[Path], output_pdf_path: Path) -> None:
    output_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    pdf = None
    try:
        for index, image_path in enumerate(image_paths):
            with Image.open(image_path) as image:
                page_width, page_height = map(float, image.size)

            if index == 0:
                pdf = canvas.Canvas(str(output_pdf_path), pagesize=(page_width, page_height))
            else:
                pdf.setPageSize((page_width, page_height))

            pdf.drawImage(
                ImageReader(str(image_path)),
                0,
                0,
                width=page_width,
                height=page_height,
            )
            pdf.showPage()

        if pdf is None:
            raise RuntimeError("No images provided to build scanned PDF.")
        pdf.save()
    finally:
        if pdf is not None:
            del pdf


def image_to_data_url(image_path: Path) -> str:
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        width, height = image.size
        scale = min(1.0, OCR_MAX_IMAGE_DIM / max(width, height))
        if scale < 1.0:
            image = image.resize((int(width * scale), int(height * scale)), Image.LANCZOS)

        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=JPEG_QUALITY, optimize=True)

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


def openai_ocr_image(image_path: Path) -> str:
    prompt = """Transcribe this page exactly into plain text.

Preserve Arabic, English, numbers, and visible symbols. Do not summarize, explain, or correct wording. Remove line breaks caused only by page width, while preserving paragraph breaks, headings, and bullet or numbered-list breaks. Return only the page text."""
    data_url = image_to_data_url(image_path)
    last_error = None

    for attempt in range(1, OPENAI_MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=OPENAI_MODEL,
                input=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": prompt},
                            {"type": "input_image", "image_url": data_url},
                        ],
                    }
                ],
            )
            return (response.output_text or "").strip()
        except Exception as error:
            last_error = error
            wait_seconds = min(20, 2 ** (attempt - 1))
            print(f"    OpenAI retry {attempt}/{OPENAI_MAX_RETRIES} for {image_path.name}: {error}")
            time.sleep(wait_seconds)

    raise RuntimeError(f"OpenAI OCR failed for {image_path.name}: {last_error}")


def ocr_images_to_txt(image_paths: list[Path], output_path: Path) -> None:
    page_texts = []
    for image_path in tqdm(image_paths, desc=f"OCR {output_path.name}", leave=False):
        page_texts.append(merge_soft_linebreaks(openai_ocr_image(image_path)).strip())
        time.sleep(SLEEP_BETWEEN_OPENAI_CALLS)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        "\n\n".join(text for text in page_texts if text).strip() + "\n",
        encoding="utf-8",
    )


def should_skip(scanned_pdf_path: Path, text_path: Path) -> bool:
    if SKIP_IF_TXT_EXISTS and text_path.exists() and not OVERWRITE_EXISTING_TXT:
        return True
    return (
        SKIP_IF_SCANNED_AND_TXT_EXIST
        and scanned_pdf_path.exists()
        and text_path.exists()
        and not OVERWRITE_EXISTING_TXT
    )


def make_temp_dir(relative_pdf_path: Path) -> Path:
    return TEMP_ROOT / relative_pdf_path.parent / relative_pdf_path.stem


def cleanup_temp_dir(temp_dir: Path) -> None:
    if CLEAN_TEMP_AFTER_EACH_FILE and temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)


pdf_files = sorted(SOURCE_PDF_ROOT.rglob(f"*{PROCESS_ONLY_EXT}"))
print(f"Found {len(pdf_files)} PDF files under {SOURCE_PDF_ROOT}")

report = {"processed": [], "skipped": [], "failed": []}

for pdf_path in pdf_files:
    relative_path = pdf_path.relative_to(SOURCE_PDF_ROOT)
    scanned_pdf_path = (SCANNED_PDF_ROOT / relative_path).with_suffix(".pdf")
    text_path = (TXT_OUTPUT_ROOT / relative_path).with_suffix(".txt")
    temp_dir = make_temp_dir(relative_path)

    if should_skip(scanned_pdf_path, text_path):
        print(f"SKIP: {relative_path}")
        report["skipped"].append(str(relative_path))
        continue

    print("=" * 100)
    print(f"PROCESSING: {relative_path}")

    try:
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)
        image_dir = temp_dir / "pages"
        image_dir.mkdir(parents=True, exist_ok=True)

        if (
            scanned_pdf_path.exists()
            and REUSE_EXISTING_SCANNED_FOR_OCR
            and (not text_path.exists() or OVERWRITE_EXISTING_TXT)
        ):
            print("  Rendering existing scanned PDF for OCR")
            page_images = render_pdf_to_jpgs(scanned_pdf_path, image_dir)
        else:
            print("  Rendering source PDF pages")
            page_images = render_pdf_to_jpgs(pdf_path, image_dir)

            if REBUILD_SCANNED_IF_MISSING or not scanned_pdf_path.exists():
                print("  Building scanned PDF")
                build_pdf_from_images(page_images, scanned_pdf_path)

        print(f"  Page count: {len(page_images)}")
        print(f"  Scanned PDF: {scanned_pdf_path}")

        if text_path.exists() and not OVERWRITE_EXISTING_TXT:
            print(f"  TXT already exists; OCR skipped: {text_path}")
        else:
            print("  OCRing page images")
            ocr_images_to_txt(page_images, text_path)
            print(f"  TXT saved: {text_path}")

        report["processed"].append(
            {
                "relative_path": str(relative_path),
                "source_pdf": str(pdf_path),
                "scanned_pdf": str(scanned_pdf_path),
                "txt_output": str(text_path),
                "page_count": len(page_images),
            }
        )
    except Exception as error:
        print(f"FAILED: {relative_path} -> {error}")
        report["failed"].append({"relative_path": str(relative_path), "error": str(error)})
    finally:
        cleanup_temp_dir(temp_dir)
        gc.collect()

with REPORT_PATH.open("w", encoding="utf-8") as report_file:
    json.dump(report, report_file, ensure_ascii=False, indent=2)

print("\nDONE")
print(f"Processed: {len(report['processed'])}")
print(f"Skipped:   {len(report['skipped'])}")
print(f"Failed:    {len(report['failed'])}")
print(f"Report:    {REPORT_PATH}")
print(f"Scanned PDFs root: {SCANNED_PDF_ROOT}")
print(f"TXT output root:   {TXT_OUTPUT_ROOT}")

### Code block 9 — Sort OCR outputs into quality-control buckets

This block classifies OCR TXT outputs into folders such as ready, review, noise, and unsafe, and copies the matching original PDFs alongside them. It supports manual review and quality control.

In [ ]:
# Install once in Colab if needed.
!pip -q install pandas

import os
import re
import shutil
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive, files

drive.mount("/content/drive")

# Configure these paths for the local Drive layout.
DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "data"))
TXT_ROOT = DRIVE_ROOT / "ocr_txt"
PDF_ROOT = DRIVE_ROOT / "source_pdfs"
REPORT_CSV = DRIVE_ROOT / "ocr_quality_report.csv"
OUTPUT_ROOT = DRIVE_ROOT / "ocr_sorted"
DELETE_OLD_OUTPUT = False

STATUSES = ("ready", "noise", "review", "unsafe")


def norm_text(value):
    if value is None:
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\\", "/")
    value = re.sub(r"/+", "/", value)
    return value.strip()


def norm_key(path_like):
    parts = [part.strip() for part in norm_text(path_like).split("/") if part.strip()]
    return "/".join(parts).casefold()


def build_index(root, suffix):
    relative_index = {}
    basename_index = {}
    for path in root.rglob(f"*{suffix}"):
        if path.is_file():
            relative_path = path.relative_to(root).as_posix()
            relative_index[norm_key(relative_path)] = path
            basename_index.setdefault(norm_key(path.name), []).append(path)
    return relative_index, basename_index


def find_by_relative_or_basename(relative_path_value, root, relative_index, basename_index, target_suffix):
    relative_path_value = norm_text(relative_path_value)
    relative_path = Path(relative_path_value)

    exact_path = root / relative_path
    if exact_path.is_file():
        return exact_path

    indexed_path = relative_index.get(norm_key(relative_path.as_posix()))
    if indexed_path is not None:
        return indexed_path

    basename = relative_path.name
    if not basename.lower().endswith(target_suffix.lower()):
        basename = relative_path.with_suffix(target_suffix).name

    matches = basename_index.get(norm_key(basename), [])
    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        # Prefer the candidate sharing the most directory components.
        relative_parts = [part.casefold() for part in relative_path.parts]
        return max(
            matches,
            key=lambda match: sum(
                part in [candidate_part.casefold() for candidate_part in match.relative_to(root).parts]
                for part in relative_parts
            ),
        )

    return None


def canonical_status(value):
    value = norm_text(value).casefold().replace("_", "-").replace(" ", "-")
    if value == "ready":
        return "ready"
    if value in {"minor-noise", "noise"}:
        return "noise"
    if value == "review":
        return "review"
    if value == "unsafe":
        return "unsafe"
    return None


def ensure_parent(path):
    path.parent.mkdir(parents=True, exist_ok=True)


def load_report_df():
    if REPORT_CSV.exists():
        print(f"Using report: {REPORT_CSV}")
        return pd.read_csv(REPORT_CSV)

    print(f"Report not found: {REPORT_CSV}")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV uploaded.")

    uploaded_name = next(iter(uploaded))
    print(f"Using uploaded report: {uploaded_name}")
    return pd.read_csv(uploaded_name)


assert TXT_ROOT.exists(), f"TXT_ROOT not found: {TXT_ROOT}"
assert PDF_ROOT.exists(), f"PDF_ROOT not found: {PDF_ROOT}"

if DELETE_OLD_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

for status in STATUSES:
    (OUTPUT_ROOT / status).mkdir(parents=True, exist_ok=True)

report_df = load_report_df()
status_col = next((column for column in report_df.columns if column.casefold().strip() == "status"), None)
file_col = next((column for column in report_df.columns if column.casefold().strip() == "file"), None)

if status_col is None:
    status_col = next(
        (
            column
            for column in report_df.columns
            if any(label in column.casefold() for label in ("status", "category", "class"))
        ),
        None,
    )

if file_col is None:
    file_col = next(
        (
            column
            for column in report_df.columns
            if column.casefold() in {"path", "filename", "relative_path", "relative path"}
            or "file" in column.casefold()
        ),
        None,
    )

assert status_col is not None, f"Could not find a status column in {report_df.columns.tolist()}"
assert file_col is not None, f"Could not find a file/path column in {report_df.columns.tolist()}"

report_df = report_df[[file_col, status_col]].copy()
report_df[file_col] = report_df[file_col].map(norm_text)
report_df[status_col] = report_df[status_col].map(canonical_status)
report_df = report_df[report_df[file_col].str.lower().str.endswith(".txt", na=False)]
report_df = report_df[report_df[status_col].isin(STATUSES)].copy()

print(f"Rows in report after filtering: {len(report_df)}")

print("Indexing TXT files...")
txt_relative_index, txt_basename_index = build_index(TXT_ROOT, ".txt")
print(f"Indexed TXT files: {len(txt_relative_index)}")

print("Indexing PDF files...")
pdf_relative_index, pdf_basename_index = build_index(PDF_ROOT, ".pdf")
print(f"Indexed PDF files: {len(pdf_relative_index)}")

manifest_rows = []
missing_txt = []
missing_pdf = []

for _, row in report_df.iterrows():
    report_txt_path = norm_text(row[file_col])
    status = row[status_col]

    txt_source = find_by_relative_or_basename(
        report_txt_path,
        TXT_ROOT,
        txt_relative_index,
        txt_basename_index,
        ".txt",
    )

    relative_txt_path = Path(report_txt_path)
    if txt_source is not None:
        relative_txt_path = txt_source.relative_to(TXT_ROOT)

    txt_destination = OUTPUT_ROOT / status / relative_txt_path
    pdf_destination = OUTPUT_ROOT / status / relative_txt_path.with_suffix(".pdf")

    txt_copied = False
    pdf_copied = False

    if txt_source is not None:
        ensure_parent(txt_destination)
        shutil.copy2(txt_source, txt_destination)
        txt_copied = True
    else:
        missing_txt.append(report_txt_path)

    relative_pdf_path = relative_txt_path.with_suffix(".pdf").as_posix()
    pdf_source = find_by_relative_or_basename(
        relative_pdf_path,
        PDF_ROOT,
        pdf_relative_index,
        pdf_basename_index,
        ".pdf",
    )

    if pdf_source is not None:
        ensure_parent(pdf_destination)
        shutil.copy2(pdf_source, pdf_destination)
        pdf_copied = True
    else:
        missing_pdf.append(relative_pdf_path)

    manifest_rows.append(
        {
            "status": status,
            "report_file": report_txt_path,
            "txt_found": txt_copied,
            "txt_source": str(txt_source) if txt_source else "",
            "txt_destination": str(txt_destination),
            "pdf_found": pdf_copied,
            "pdf_source": str(pdf_source) if pdf_source else "",
            "pdf_destination": str(pdf_destination),
        }
    )

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_ROOT / "copy_manifest.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

missing_txt_path = OUTPUT_ROOT / "missing_txt.csv"
pd.DataFrame({"missing_txt": missing_txt}).to_csv(
    missing_txt_path,
    index=False,
    encoding="utf-8-sig",
)

missing_pdf_path = OUTPUT_ROOT / "missing_pdf.csv"
pd.DataFrame({"missing_pdf": missing_pdf}).to_csv(
    missing_pdf_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Output root: {OUTPUT_ROOT}")
print(f"Manifest: {manifest_path}")
print(f"Missing TXT: {missing_txt_path}")
print(f"Missing PDF: {missing_pdf_path}")

summary = manifest_df.groupby("status")[["txt_found", "pdf_found"]].sum()
summary = summary.join(manifest_df.groupby("status").size().rename("total_rows"))
print("\nSummary:")
print(summary)

print("\nCreated category folders:")
for status in STATUSES:
    category_path = OUTPUT_ROOT / status
    txt_count = sum(1 for _ in category_path.rglob("*.txt"))
    pdf_count = sum(1 for _ in category_path.rglob("*.pdf"))
    print(f"  {status}: {txt_count} txt | {pdf_count} pdf")

### Code block 10 — Trim the OCR Arabic TXT corpus with the OpenAI API

This block removes unwanted material such as tables of contents or references from the OCR outputs while preserving meaningful body text. The cleaned texts become better suited for AI-detection analysis.

In [ ]:
import json
import os
import re
import shutil
import time
import unicodedata
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# Configure paths and credentials through the environment.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-5")
INPUT_ROOT = Path(os.environ.get("ARABIC_TXT_INPUT_ROOT", "input_txt")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("ARABIC_TXT_OUTPUT_ROOT", "trimmed_txt")).expanduser()

KEPT_ROOT = OUTPUT_ROOT / "kept"
REMOVED_ROOT = OUTPUT_ROOT / "removed"
LOG_CSV = OUTPUT_ROOT / "trim_log.csv"
REMOVED_CSV = OUTPUT_ROOT / "removed_under_minimum_words.csv"
ERROR_CSV = OUTPUT_ROOT / "errors.csv"

DELETE_OLD_OUTPUT = False
MIN_WORDS_AFTER_TRIM = 300
HEAD_CHAR_LIMIT = 18_000
TAIL_CHAR_LIMIT = 18_000
MAX_RETRIES = 4
RETRY_SLEEP_SEC = 4
SLEEP_BETWEEN_FILES_SEC = 0.8

if not OPENAI_API_KEY:
    raise ValueError("Set the OPENAI_API_KEY environment variable.")
if not INPUT_ROOT.exists():
    raise FileNotFoundError(f"Input directory not found: {INPUT_ROOT}")

client = OpenAI(api_key=OPENAI_API_KEY)

if DELETE_OLD_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

for directory in (KEPT_ROOT, REMOVED_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

TATWEEL_RE = re.compile(r"ـ+")


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\ufeff", "")
    text = text.replace("\u200f", "").replace("\u200e", "")
    text = text.replace("\xa0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = TATWEEL_RE.sub("", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def count_words_ar_mixed(text: str) -> int:
    return len(re.findall(r"[\u0600-\u06FFA-Za-z0-9_]+", text))


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def read_txt(path: Path) -> str:
    for encoding in ("utf-8", "utf-8-sig", "cp1256", "latin-1"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_bytes().decode("utf-8", errors="ignore")


def write_txt(path: Path, text: str) -> None:
    ensure_parent(path)
    path.write_text(text, encoding="utf-8")


def split_head_tail(text: str, head_limit: int, tail_limit: int) -> Tuple[str, str]:
    # Only the document boundaries are inspected to avoid sending body text unnecessarily.
    if len(text) <= head_limit + tail_limit:
        return text, text
    return text[:head_limit], text[-tail_limit:]


def safe_json_extract(text: str) -> dict:
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
    raise ValueError("Could not parse JSON from model output.")


def extract_lines(text: str) -> List[str]:
    return text.splitlines()


def char_index_from_line_no(text: str, line_no_1_based: int) -> int:
    if line_no_1_based <= 1:
        return 0

    lines = extract_lines(text)
    line_no_1_based = min(max(1, line_no_1_based), len(lines) + 1)
    return sum(len(lines[index]) + 1 for index in range(line_no_1_based - 1))


def apply_line_boundary_trim(
    full_text: str,
    front_end_line: Optional[int],
    back_start_line_in_tail: Optional[int],
    tail_text: str,
) -> str:
    text = full_text

    if isinstance(front_end_line, int) and front_end_line > 0:
        front_cut = char_index_from_line_no(text, front_end_line + 1)
        text = text[front_cut:].lstrip()

    if isinstance(back_start_line_in_tail, int) and back_start_line_in_tail > 0:
        tail_position = full_text.rfind(tail_text)
        if tail_position != -1:
            tail_lines = extract_lines(tail_text)
            back_start_line_in_tail = min(back_start_line_in_tail, len(tail_lines))
            cut_in_tail = char_index_from_line_no(tail_text, back_start_line_in_tail)
            back_cut = tail_position + cut_in_tail

            front_trimmed_chars = (
                len(full_text) - len(text) if full_text.endswith(text) else None
            )
            text = full_text[:back_cut].rstrip()
            if front_trimmed_chars is not None:
                text = text[front_trimmed_chars:].lstrip()

    return normalize_text(text)


def detect_trim_boundaries_with_openai(head_text: str, tail_text: str) -> dict:
    prompt = f"""
افحص بداية ونهاية النص العربي لتحديد حدود الأقسام التمهيدية والختامية القابلة للحذف.

افحص البداية فقط لتحديد ما إذا كانت تحتوي على فهرس أو جدول محتويات أو قائمة عناوين مع أرقام صفحات.
افحص النهاية فقط لتحديد ما إذا كانت تحتوي على مراجع أو مصادر أو قائمة استشهادات.
لا تحذف متن الوثيقة، العناوين العادية، المقدمة، الخاتمة، التوصيات، أو ذكر المصادر داخل الفقرات.
إذا لم تكن متأكدًا، اختر عدم الحذف.

أعد JSON فقط بالشكل التالي:
{{
  "remove_front": true,
  "front_end_line": 0,
  "front_reason": "short",
  "remove_back": false,
  "back_start_line": null,
  "back_reason": "short",
  "confidence": 0.0
}}

القواعد:
- front_end_line هو آخر سطر من الفهرس المراد حذفه من البداية، ويكون null عند عدم الحذف.
- back_start_line هو أول سطر من المراجع المراد حذفه من مقتطف النهاية، ويكون null عند عدم الحذف.
- قد تظهر المراجع بصيغ مثل: المراجع، قائمة المراجع، المصادر، References، Bibliography.
- قد يظهر الفهرس بصيغ مثل: الفهرس، المحتويات، جدول المحتويات، أو عناوين قصيرة متبوعة بأرقام صفحات.

بداية النص:
<<<HEAD>>>
{head_text}
<<<END_HEAD>>>

نهاية النص:
<<<TAIL>>>
{tail_text}
<<<END_TAIL>>>
"""

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(model=MODEL_NAME, input=prompt)
            decision = safe_json_extract(response.output_text)
            decision.setdefault("remove_front", False)
            decision.setdefault("front_end_line", None)
            decision.setdefault("front_reason", "")
            decision.setdefault("remove_back", False)
            decision.setdefault("back_start_line", None)
            decision.setdefault("back_reason", "")
            decision.setdefault("confidence", None)
            return decision
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP_SEC * attempt)

    raise last_error


def trim_one_file(text: str) -> Tuple[str, dict]:
    original = normalize_text(text)
    head, tail = split_head_tail(original, HEAD_CHAR_LIMIT, TAIL_CHAR_LIMIT)
    decision = detect_trim_boundaries_with_openai(head, tail)

    trimmed = apply_line_boundary_trim(
        full_text=original,
        front_end_line=decision["front_end_line"] if decision["remove_front"] else None,
        back_start_line_in_tail=(
            decision["back_start_line"] if decision["remove_back"] else None
        ),
        tail_text=tail,
    )

    metadata = {
        "remove_front": bool(decision["remove_front"]),
        "front_end_line": decision["front_end_line"],
        "front_reason": decision["front_reason"],
        "remove_back": bool(decision["remove_back"]),
        "back_start_line": decision["back_start_line"],
        "back_reason": decision["back_reason"],
        "confidence": decision["confidence"],
        "original_chars": len(original),
        "trimmed_chars": len(trimmed),
        "original_words": count_words_ar_mixed(original),
        "trimmed_words": count_words_ar_mixed(trimmed),
    }
    return trimmed, metadata


txt_files = sorted(path for path in INPUT_ROOT.rglob("*.txt") if path.is_file())
if not txt_files:
    raise FileNotFoundError(f"No TXT files found under {INPUT_ROOT}")

log_rows = []
removed_rows = []
error_rows = []

print(f"Found {len(txt_files)} TXT files.")

for txt_path in tqdm(txt_files, desc="Processing TXT files"):
    relative_path = txt_path.relative_to(INPUT_ROOT)

    try:
        raw = normalize_text(read_txt(txt_path))

        if not raw:
            output_path = REMOVED_ROOT / relative_path
            write_txt(output_path, "")
            removed_rows.append(
                {
                    "file": relative_path.as_posix(),
                    "reason": "empty_after_read",
                    "original_words": 0,
                    "trimmed_words": 0,
                }
            )
            log_rows.append(
                {
                    "file": relative_path.as_posix(),
                    "status": "removed",
                    "remove_front": False,
                    "front_end_line": None,
                    "front_reason": "",
                    "remove_back": False,
                    "back_start_line": None,
                    "back_reason": "",
                    "confidence": None,
                    "original_chars": 0,
                    "trimmed_chars": 0,
                    "original_words": 0,
                    "trimmed_words": 0,
                    "output_path": str(output_path),
                }
            )
            continue

        trimmed, metadata = trim_one_file(raw)
        if metadata["trimmed_words"] < MIN_WORDS_AFTER_TRIM:
            output_path = REMOVED_ROOT / relative_path
            write_txt(output_path, trimmed)
            removed_rows.append(
                {
                    "file": relative_path.as_posix(),
                    "reason": f"under_{MIN_WORDS_AFTER_TRIM}_words_after_trim",
                    "original_words": metadata["original_words"],
                    "trimmed_words": metadata["trimmed_words"],
                }
            )
            status = "removed"
        else:
            output_path = KEPT_ROOT / relative_path
            write_txt(output_path, trimmed)
            status = "kept"

        log_rows.append(
            {
                "file": relative_path.as_posix(),
                "status": status,
                **metadata,
                "output_path": str(output_path),
            }
        )
        time.sleep(SLEEP_BETWEEN_FILES_SEC)

    except Exception as error:
        error_rows.append({"file": relative_path.as_posix(), "error": str(error)})
        log_rows.append(
            {
                "file": relative_path.as_posix(),
                "status": "error",
                "remove_front": None,
                "front_end_line": None,
                "front_reason": "",
                "remove_back": None,
                "back_start_line": None,
                "back_reason": "",
                "confidence": None,
                "original_chars": None,
                "trimmed_chars": None,
                "original_words": None,
                "trimmed_words": None,
                "output_path": "",
            }
        )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(log_rows).to_csv(LOG_CSV, index=False, encoding="utf-8-sig")
pd.DataFrame(removed_rows).to_csv(REMOVED_CSV, index=False, encoding="utf-8-sig")
pd.DataFrame(error_rows).to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")

print(f"Kept files: {sum(1 for _ in KEPT_ROOT.rglob('*.txt'))}")
print(f"Removed files: {sum(1 for _ in REMOVED_ROOT.rglob('*.txt'))}")
print(f"Errors: {len(error_rows)}")
print(f"Logs: {LOG_CSV}")

### Code block 11 — Run AI detectors on the Arabic TXT corpus

This block sends the cleaned Arabic TXT files through GPTZero, Pangram, Isgen, and Sapling, producing one detector row per file for later comparison with the English and code branches.

In [ ]:
%pip -q install pandas numpy requests

import json
import os
import re
import time

import numpy as np
import pandas as pd
import requests

# Configure private data and credentials outside the notebook.
INPUT_ROOT = os.environ.get("ARABIC_TXT_INPUT_ROOT", "")
OUTPUT_DIR = os.environ.get("ARABIC_DETECTION_OUTPUT_DIR", INPUT_ROOT)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "arabic_ai_detection_results.csv")
BACKUP_JSONL = os.path.join(OUTPUT_DIR, "arabic_ai_detection_log.jsonl")
BACKUP_TXT = os.path.join(OUTPUT_DIR, "arabic_ai_detection_log.txt")

GPTZERO_API_KEY = os.environ.get("GPTZERO_API_KEY", "")
PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY", "")
SAPLING_API_KEY = os.environ.get("SAPLING_API_KEY", "")
ISGEN_RAPIDAPI_KEY = os.environ.get("ISGEN_RAPIDAPI_KEY", "")

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180
RESUME_IF_OUTPUT_EXISTS = True
MAX_TEXT_CHARS = 50000


def safe_str(value):
    return "" if pd.isna(value) else str(value)


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text)) if text else 0


def read_txt(path):
    for encoding in ["utf-8", "utf-8-sig", "cp1256", "cp1252", "latin-1"]:
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""

    # Preserve Arabic text and punctuation; remove only transport/control artifacts.
    text = text.replace("\x00", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("\u200f", "").replace("\u200e", "")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if not text or len(text) <= max_chars:
        return text or ""

    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_boundary = max(
        cut.rfind(". "),
        cut.rfind("! "),
        cut.rfind("? "),
        cut.rfind("؟ "),
        cut.rfind("۔ "),
        cut.rfind("\n"),
    )
    if last_boundary > max_chars * 0.6:
        return cut[: last_boundary + 1].strip()

    return cut.strip()


def detect_language_simple(text):
    if not text:
        return "en"

    arabic_chars = re.findall(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]", text)
    return "ar" if len(arabic_chars) / len(text) > 0.03 else "en"


def request_with_retries(method, url, **kwargs):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(method, url, timeout=REQUEST_TIMEOUT, **kwargs)
            if 200 <= response.status_code < 300:
                return response
            last_error = RuntimeError(f"HTTP {response.status_code}: {response.text[:1200]}")
        except requests.RequestException as error:
            last_error = error

        if attempt < MAX_RETRIES:
            time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    raise last_error


def save_progress(df):
    df = df[FINAL_COLUMNS].sort_values("relative_path").reset_index(drop=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    return df


def append_jsonl(record):
    with open(BACKUP_JSONL, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_txt(line):
    with open(BACKUP_TXT, "a", encoding="utf-8") as file:
        file.write(line + "\n")


def ai_vs_human_label_from_text(value):
    value = safe_str(value).strip().lower()
    if not value:
        return ""

    if any(term in value for term in ["mixed", "ai-assisted", "assisted"]):
        return "AI"
    if any(term in value for term in ["ai", "likely ai", "generated", "machine", "synthetic", "written by ai", "contains ai", "predominantly ai"]):
        return "AI"
    if any(term in value for term in ["human", "likely human", "written by human", "human-written", "authored by human"]):
        return "Human"
    return value


def log_tool_result(relative_path, file_name, word_count, chars_sent, tool, ai_rate, result, status, error=None, raw_payload=None):
    record = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "relative_path": relative_path,
        "file_name": file_name,
        "tool": tool,
        "word_count": word_count,
        "chars_sent_to_api": chars_sent,
        "status": status,
        "ai_rate_percent": None if pd.isna(ai_rate) else ai_rate,
        "main_result": result,
        "error": error,
        "raw_payload": raw_payload,
    }
    append_jsonl(record)
    append_txt(
        f"[{record['timestamp']}] path={relative_path} | file={file_name} | "
        f"tool={tool} | status={status} | word_count={word_count} | "
        f"chars_sent={chars_sent} | ai_rate_percent={record['ai_rate_percent']} | "
        f"main_result={result} | error={error or ''}"
    )


def row_index(df, relative_path):
    matches = df.index[df["relative_path"].astype(str) == str(relative_path)]
    return matches[0] if len(matches) else None


def get_or_create_row_idx(df, relative_path, file_name):
    existing_idx = row_index(df, relative_path)
    if existing_idx is not None:
        return existing_idx

    new_idx = len(df)
    df.loc[new_idx, "relative_path"] = relative_path
    df.loc[new_idx, "file_name"] = file_name
    return new_idx


def call_gptzero(text, api_key):
    response = request_with_retries(
        "POST",
        "https://api.gptzero.me/v2/predict/text",
        headers={
            "x-api-key": api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
        json={"document": text},
    )
    data = response.json()
    document = data.get("documents", [data])[0] if isinstance(data, dict) else None
    if not isinstance(document, dict):
        raise RuntimeError(f"GPTZero unexpected response shape: {str(data)[:500]}")

    probabilities = document.get("class_probabilities", {}) or {}
    return {
        "raw": data,
        "ai_score": probabilities.get("ai"),
        "label": document.get("predicted_class") or document.get("document_classification"),
    }


def call_pangram(text, api_key):
    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={"Content-Type": "application/json", "x-api-key": api_key},
        json={"text": text, "public_dashboard_link": False},
    )
    data = response.json()
    return {
        "raw": data,
        "ai_score": data.get("fraction_ai"),
        "label": data.get("prediction_short") or data.get("prediction"),
    }


def call_sapling(text, api_key):
    response = request_with_retries(
        "POST",
        "https://api.sapling.ai/api/v1/aidetect",
        json={"key": api_key, "text": text},
    )
    data = response.json()
    return {"raw": data, "ai_score": data.get("score"), "label": ""}


def call_isgen(text, lang, api_key):
    response = request_with_retries(
        "POST",
        "https://ai-detection4.p.rapidapi.com/v1/ai-detection-rapid-api",
        headers={
            "x-rapidapi-key": api_key,
            "x-rapidapi-host": "ai-detection4.p.rapidapi.com",
            "Content-Type": "application/json",
        },
        json={"text": text, "lang": lang},
    )
    data = response.json()
    summary = data.get("summary", {}) or {}
    return {
        "raw": data,
        "ai_score": data.get("ai_score"),
        "summary_ai": summary.get("ai"),
        "summary_human": summary.get("human"),
        "summary_mixed": summary.get("mixed"),
    }


def normalize_standard_result(result):
    ai_rate = normalize_percent(result.get("ai_score"))
    label = ai_vs_human_label_from_text(result.get("label"))
    if not label and pd.notna(ai_rate):
        label = "AI" if ai_rate >= 50 else "Human"
    return ai_rate, label


def normalize_isgen_result(result):
    ai_rate = normalize_percent(result.get("ai_score"))
    labels = {
        "AI": normalize_percent(result.get("summary_ai")),
        "Human": normalize_percent(result.get("summary_human")),
    }
    mixed_score = normalize_percent(result.get("summary_mixed"))
    if pd.notna(mixed_score):
        labels["AI"] = mixed_score

    labels = {label: score for label, score in labels.items() if pd.notna(score)}
    if labels:
        return ai_rate, max(labels, key=labels.get)
    if pd.notna(ai_rate):
        return ai_rate, "AI" if ai_rate >= 50 else "Human"
    return np.nan, ""


FINAL_COLUMNS = [
    "relative_path",
    "file_name",
    "word_count",
    "chars_sent_to_api",
    "detected_language",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
]

if not INPUT_ROOT or not os.path.isdir(INPUT_ROOT):
    raise FileNotFoundError("Set ARABIC_TXT_INPUT_ROOT to an existing directory of TXT files.")

if not OUTPUT_DIR:
    raise ValueError("Set ARABIC_DETECTION_OUTPUT_DIR or ARABIC_TXT_INPUT_ROOT.")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_txt_files = sorted(
    (
        (os.path.relpath(os.path.join(root, name), INPUT_ROOT).replace("\\", "/"), os.path.join(root, name))
        for root, _, files in os.walk(INPUT_ROOT)
        for name in files
        if name.lower().endswith(".txt")
    ),
    key=lambda item: item[0].lower(),
)

if not all_txt_files:
    raise ValueError(f"No TXT files found under: {INPUT_ROOT}")

if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_CSV):
    out = pd.read_csv(OUTPUT_CSV)
    if "relative_path" not in out.columns:
        out = pd.DataFrame(columns=FINAL_COLUMNS)
else:
    out = pd.DataFrame(columns=FINAL_COLUMNS)

for column in FINAL_COLUMNS:
    if column not in out.columns:
        out[column] = np.nan if column in {"word_count", "chars_sent_to_api"} or column.endswith("_ai_rate_percent") else ""
out = out[FINAL_COLUMNS]

TOOLS = [
    ("gptzero", "GPTZero", GPTZERO_API_KEY, lambda text, lang, key: call_gptzero(text, key), normalize_standard_result),
    ("pangram", "Pangram", PANGRAM_API_KEY, lambda text, lang, key: call_pangram(text, key), normalize_standard_result),
    ("sapling", "Sapling", SAPLING_API_KEY, lambda text, lang, key: call_sapling(text, key), normalize_standard_result),
    ("isgen", "Isgen", ISGEN_RAPIDAPI_KEY, call_isgen, normalize_isgen_result),
]

failure_log = []

for file_number, (relative_path, txt_path) in enumerate(all_txt_files, start=1):
    file_name = os.path.basename(txt_path)
    print(f"[{file_number}/{len(all_txt_files)}] {relative_path}")

    row_idx = get_or_create_row_idx(out, relative_path, file_name)
    cleaned_text = clean_text(read_txt(txt_path))
    text_for_api = truncate_text(cleaned_text)
    language = detect_language_simple(text_for_api)
    word_count = count_words(cleaned_text)

    out.at[row_idx, "file_name"] = file_name
    out.at[row_idx, "word_count"] = word_count
    out.at[row_idx, "chars_sent_to_api"] = len(text_for_api)
    out.at[row_idx, "detected_language"] = language
    out = save_progress(out)
    row_idx = row_index(out, relative_path)

    if not text_for_api.strip():
        for tool_key, _, _, _, _ in TOOLS:
            out.at[row_idx, f"{tool_key}_ai_rate_percent"] = np.nan
            out.at[row_idx, f"{tool_key}_result"] = "ERROR"
        out = save_progress(out)
        print("  Empty text after cleaning; marked all tools as ERROR.")
        continue

    print(f"  words={word_count} | lang={language} | chars_sent={len(text_for_api)}")
    for tool_key, tool_name, api_key, call_tool, normalize_result in TOOLS:
        try:
            result = call_tool(text_for_api, language, api_key)
            ai_rate, main_result = normalize_result(result)
            out.at[row_idx, f"{tool_key}_ai_rate_percent"] = ai_rate
            out.at[row_idx, f"{tool_key}_result"] = main_result
            log_tool_result(relative_path, file_name, word_count, len(text_for_api), tool_name, ai_rate, main_result, "OK", raw_payload=result.get("raw"))
            print(f"    {tool_name}: ai_rate_percent={ai_rate} | result={main_result}")
        except Exception as error:
            out.at[row_idx, f"{tool_key}_ai_rate_percent"] = np.nan
            out.at[row_idx, f"{tool_key}_result"] = "ERROR"
            failure_log.append(f"{relative_path} | {tool_name} | {error}")
            log_tool_result(relative_path, file_name, word_count, len(text_for_api), tool_name, np.nan, "ERROR", "FAILED", error=str(error))
            print(f"    {tool_name} failed: {error}")

        out = save_progress(out)
        row_idx = row_index(out, relative_path)
        time.sleep(SLEEP_BETWEEN_CALLS)

out = save_progress(out)
print(f"Completed: {len(out)} files")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Failure count: {len(failure_log)}")

### Code block 12 — Retry only the Pangram failures

This block reruns Pangram on the small subset of Arabic files that failed previously and writes the repaired results back into the existing result set instead of rerunning the whole Arabic corpus.

In [ ]:
import json
import os
import re
import time

import numpy as np
import pandas as pd
import requests


def require_env(name):
    value = os.environ.get(name)
    if not value:
        raise EnvironmentError(f"Set the {name} environment variable.")
    return value


INPUT_ROOT = require_env("INPUT_ROOT")
OUTPUT_CSV = require_env("OUTPUT_CSV")
BACKUP_JSONL = require_env("BACKUP_JSONL")
BACKUP_TXT = require_env("BACKUP_TXT")
PANGRAM_API_KEY = require_env("PANGRAM_API_KEY")

# List of relative file paths (JSON array) for which previous runs failed.
FAILED_RELATIVE_PATHS = json.loads(require_env("FAILED_RELATIVE_PATHS_JSON"))
if not isinstance(FAILED_RELATIVE_PATHS, list) or not all(
    isinstance(path, str) for path in FAILED_RELATIVE_PATHS
):
    raise ValueError("FAILED_RELATIVE_PATHS_JSON must contain a JSON array of strings.")

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 3
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180
MAX_TEXT_CHARS = 50000


def safe_str(value):
    return "" if pd.isna(value) else str(value)


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text)) if text else 0


def read_txt(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1256", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("\u200f", "").replace("\u200e", "")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if not text or len(text) <= max_chars:
        return text or ""

    # Prefer returning a complete paragraph or sentence when possible to avoid
    # truncation mid-sentence; fall back to a hard cut otherwise.
    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_boundary = max(
        cut.rfind(". "),
        cut.rfind("! "),
        cut.rfind("? "),
        cut.rfind("؟ "),
        cut.rfind("۔ "),
        cut.rfind("\n"),
    )
    if last_boundary > max_chars * 0.6:
        return cut[: last_boundary + 1].strip()

    return cut.strip()


def request_with_retries(method, url, **kwargs):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(method, url, timeout=REQUEST_TIMEOUT, **kwargs)
            if 200 <= response.status_code < 300:
                return response
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:1200]}"
            )
        except requests.RequestException as error:
            last_error = error

        if attempt < MAX_RETRIES:
            time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    raise last_error


def append_jsonl(record, path):
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_txt(line, path):
    with open(path, "a", encoding="utf-8") as file:
        file.write(line + "\n")


def save_progress(dataframe, output_csv):
    # Persist a sorted copy to keep on-disk CSV consistent; do not reorder
    # the in-memory dataframe which may rely on current row indices.
    dataframe.sort_values("relative_path").to_csv(
        output_csv, index=False, encoding="utf-8-sig"
    )


def log_tool_result(
    relative_path,
    file_name,
    word_count_value,
    chars_sent,
    tool_name,
    ai_rate,
    main_result,
    status,
    error_msg=None,
    raw_payload=None,
):
    record = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "relative_path": relative_path,
        "file_name": file_name,
        "tool": tool_name,
        "word_count": word_count_value,
        "chars_sent_to_api": chars_sent,
        "status": status,
        "ai_rate_percent": None if pd.isna(ai_rate) else ai_rate,
        "main_result": main_result,
        "error": error_msg,
        "raw_payload": raw_payload,
    }
    append_jsonl(record, BACKUP_JSONL)

    append_txt(
        f"[{record['timestamp']}] "
        f"path={relative_path} | file={file_name} | tool={tool_name} | "
        f"status={status} | word_count={word_count_value} | "
        f"chars_sent={chars_sent} | "
        f"ai_rate_percent={record['ai_rate_percent']} | "
        f"main_result={main_result} | error={error_msg or ''}",
        BACKUP_TXT,
    )


def call_pangram(text, api_key):
    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={"Content-Type": "application/json", "x-api-key": api_key},
        json={"text": text, "public_dashboard_link": False},
    )
    data = response.json()

    return {
        "raw": data,
        "pangram_prediction_short": data.get("prediction_short"),
        "pangram_prediction": data.get("prediction"),
        "pangram_headline": data.get("headline"),
        "pangram_fraction_ai": data.get("fraction_ai"),
        "pangram_fraction_ai_assisted": data.get("fraction_ai_assisted"),
        "pangram_fraction_human": data.get("fraction_human"),
        "pangram_num_ai_segments": data.get("num_ai_segments"),
        "pangram_num_ai_assisted_segments": data.get("num_ai_assisted_segments"),
        "pangram_num_human_segments": data.get("num_human_segments"),
        "pangram_num_windows": len(data.get("windows", []) or []),
    }


def ai_vs_human_label_from_text(value):
    value = safe_str(value).strip().lower()
    if not value:
        return ""

    ai_keywords = (
        "ai",
        "likely ai",
        "generated",
        "machine",
        "synthetic",
        "written by ai",
        "contains ai",
        "predominantly ai",
    )
    human_keywords = (
        "human",
        "likely human",
        "written by human",
        "human-written",
        "authored by human",
    )
    mixed_keywords = ("mixed", "ai-assisted", "assisted")

    if any(keyword in value for keyword in mixed_keywords + ai_keywords):
        return "AI"
    if any(keyword in value for keyword in human_keywords):
        return "Human"
    return value


def normalize_pangram_result(result):
    ai_rate = normalize_percent(result.get("pangram_fraction_ai"))
    raw_label = (
        result.get("pangram_prediction_short") or result.get("pangram_prediction") or ""
    )
    # If the label is empty but a numeric AI fraction exists, derive label from ai_rate.
    if not safe_str(raw_label).strip() and pd.notna(ai_rate):
        raw_label = "AI" if ai_rate >= 50 else "Human"

    label = ai_vs_human_label_from_text(raw_label)
    if not label and pd.notna(ai_rate):
        label = "AI" if ai_rate >= 50 else "Human"

    return ai_rate, label


if not os.path.exists(OUTPUT_CSV):
    raise FileNotFoundError(f"Missing output CSV: {OUTPUT_CSV}")

out = pd.read_csv(OUTPUT_CSV)
required_columns = (
    "relative_path",
    "file_name",
    "word_count",
    "chars_sent_to_api",
    "pangram_ai_rate_percent",
    "pangram_result",
)

for column in required_columns:
    if column not in out.columns:
        out[column] = (
            np.nan
            if column in {"word_count", "chars_sent_to_api", "pangram_ai_rate_percent"}
            else ""
        )

failure_log = []
print(f"Retrying Pangram for {len(FAILED_RELATIVE_PATHS)} files")

for index, relative_path in enumerate(FAILED_RELATIVE_PATHS, start=1):
    txt_path = os.path.join(INPUT_ROOT, relative_path)
    file_name = os.path.basename(txt_path)

    print(f"[{index}/{len(FAILED_RELATIVE_PATHS)}] {relative_path}")

    if not os.path.exists(txt_path):
        message = f"Missing TXT file: {txt_path}"
        failure_log.append(message)
        print(f"  FAILED -> {message}")
        continue

    matches = out.index[out["relative_path"].astype(str) == str(relative_path)].tolist()
    if matches:
        row_index = matches[0]
    else:
        row_index = len(out)
        out.loc[row_index, "relative_path"] = relative_path
        out.loc[row_index, "file_name"] = file_name

    cleaned_text = clean_text(read_txt(txt_path))
    text_for_api = truncate_text(cleaned_text)
    word_count = count_words(cleaned_text)

    out.at[row_index, "relative_path"] = relative_path
    out.at[row_index, "file_name"] = file_name
    out.at[row_index, "word_count"] = word_count
    out.at[row_index, "chars_sent_to_api"] = len(text_for_api)
    save_progress(out, OUTPUT_CSV)

    print(f"  RUNNING Pangram | words={word_count} | chars_sent={len(text_for_api)}")

    if not text_for_api.strip():
        message = "Empty text after cleaning/truncation"
        out.at[row_index, "pangram_ai_rate_percent"] = np.nan
        out.at[row_index, "pangram_result"] = "ERROR"
        save_progress(out, OUTPUT_CSV)
        failure_log.append(f"{relative_path} | Pangram | {message}")
        print(f"  FAILED -> {message}")
        log_tool_result(
            relative_path,
            file_name,
            word_count,
            len(text_for_api),
            "Pangram",
            np.nan,
            "ERROR",
            "FAILED",
            error_msg=message,
        )
        continue

    try:
        result = call_pangram(text_for_api, PANGRAM_API_KEY)
        ai_rate, main_result = normalize_pangram_result(result)

        out.at[row_index, "pangram_ai_rate_percent"] = ai_rate
        out.at[row_index, "pangram_result"] = main_result
        save_progress(out, OUTPUT_CSV)

        print(f"  Pangram OK -> ai_rate_percent={ai_rate} | result={main_result}")
        log_tool_result(
            relative_path,
            file_name,
            word_count,
            len(text_for_api),
            "Pangram",
            ai_rate,
            main_result,
            "OK",
            raw_payload=result.get("raw"),
        )
    except Exception as error:
        out.at[row_index, "pangram_ai_rate_percent"] = np.nan
        out.at[row_index, "pangram_result"] = "ERROR"
        save_progress(out, OUTPUT_CSV)

        failure_log.append(f"{relative_path} | Pangram | {error}")
        print(f"  Pangram FAILED -> {error}")
        log_tool_result(
            relative_path,
            file_name,
            word_count,
            len(text_for_api),
            "Pangram",
            np.nan,
            "ERROR",
            "FAILED",
            error_msg=str(error),
        )

    time.sleep(SLEEP_BETWEEN_CALLS)

save_progress(out, OUTPUT_CSV)

print("DONE")
print("Updated CSV:", OUTPUT_CSV)
print("Backup JSONL:", BACKUP_JSONL)
print("Backup TXT:", BACKUP_TXT)
print("Remaining failure count:", len(failure_log))

if failure_log:
    print("Failures:")
    for failure in failure_log:
        print("-", failure)

show_columns = (
    "relative_path",
    "word_count",
    "chars_sent_to_api",
    "pangram_ai_rate_percent",
    "pangram_result",
)
updated_rows = out.loc[
    out["relative_path"].isin(FAILED_RELATIVE_PATHS), show_columns
].sort_values("relative_path")
print("Updated Pangram rows:")
print(updated_rows.to_string(index=False))